# 🔮🧠🤖 <span style="color: white; background-color: steelblue;"><b> Modelo de Previsão de Turnover - UL </b></span></p>

# 📑 <span style="color: LightSeaGreen;"><i> Funcionalidades Principais: </i></span></p>

📁 <span style="color: mediumseagreen;"><b> SEÇÃO 1-2: CARREGAMENTO E PREPARAÇÃO DE DADOS </b></span></p>
- Pré-processamento robusto com tratamento de valores faltantes
- Conversão automática de tipos de dados
- Identificação e remoção de data leakage (status → target)
- Remoção de variáveis sem poder preditivo (cep, cota_pcd, deficiente)
- Geração de features derivadas (idade, dias_trabalhado, etc.)

🔍 <span style="color: mediumseagreen;"><b> SEÇÃO 3: ANÁLISE EXPLORATÓRIA (EDA) </b></span></p>
- Estatísticas descritivas de todas as variáveis
- Heatmap de correlação
- Histogramas e Boxplots para 7 variáveis numéricas
- Countplots categorizados para 4 variáveis categóricas
- Análise da distribuição da variável alvo (balanceamento de classes)

🧠 <span style="color: mediumseagreen;"><b> SEÇÃO 4.1.A: SELEÇÃO AUTOMÁTICA DE FEATURES (INOVAÇÃO) </b></span></p>
- Teste Chi-Square para variáveis categóricas (p-value < 0.05)
- Teste F (ANOVA) para variáveis numéricas (p-value < 0.05)
- Cálculo e remoção de multicolinearidade (VIF > 5.0)
- Identificação de features removidas por cada critério
- logger detalhado de todas as remoções
  - Resultado: Features dinâmicas que se adaptam aos dados

🤖 <span style="color: mediumseagreen;"><b> SEÇÃO 4.2: PREPARAÇÃO DE FEATURES PARA SCIKIT-LEARN </b></span></p>
- StandardScaler para variáveis numéricas
- OneHotEncoder para variáveis categóricas
- ColumnTransformer para pipeline integrado
- Preservação de 'empresa_resumo' (padrões por unidade)

🧪 <span style="color: mediumseagreen;"><b> SEÇÃO 4.3-4.4: CONSTRUÇÃO E AVALIAÇÃO DO MODELO SCIKIT-LEARN </b></span></p>
- Pipeline com LogisticRegression (class_weight='balanced')
- Divisão treino/teste com estratificação (80/20)
- Validação cruzada (StratifiedKFold com 5 folds)
- Ajuste dinâmico do threshold para otimizar F1-Score
- Métricas completas: Acurácia, Precisão, Recall, F1-Score, ROC AUC, Gini
- Matriz de Confusão com visualização
- Curva ROC com Coeficiente de Gini

📐 <span style="color: mediumseagreen;"><b> SEÇÃO 4.5.B: CONSTRUÇÃO E ANÁLISE DO GLM (STATSMODELS) </b></span></p>
- Preprocessamento exclusivo para GLM:
  - Remove 'empresa_resumo' (multicolinearidade)
  - Consolidação de categorias raras (min_freq = 5%)
- Remoção iterativa de features com VIF > 5.0
- Detecção e remoção de separação perfeita
- PASSO 6.5: Rastreamento completo de quais features foram removidas
  - Por VIF alto
  - Por separação perfeita
  - Análise específica de 'formacoes'
- Extração de coeficientes, Odds Ratios e p-values
- Identificação de fatores de risco/proteção com significância estatística

📤 <span style="color: mediumseagreen;"><b> SEÇÃO 5: PREVISÕES E EXPORTAÇÃO DE RESULTADOS </b></span></p>
- Cálculo de probabilidades para todos os colaboradores
- Segmentação automática por risco:
  - Baixo Risco: Prob < 30%
  - Médio Risco: 30% ≤ Prob < 70%
  - Alto Risco: Prob ≥ 70%
- Exportação para Excel com formatação profissional
- Incluição de todas as variáveis originais + previsões

🧩 <span style="color: mediumseagreen;"><b> SEÇÃO 6: RECOMENDAÇÕES ESTRATÉGICAS </b></span></p>
- Ações específicas por segmento de risco
- KPIs de acompanhamento para RH
- Alertas para colaboradores em alto risco

🌐 <span style="color: mediumseagreen;"><b> SEÇÃO 7: RELATÓRIO HTML INTERATIVO </b></span></p>
- Resumo executivo com métricas principais
- Contextualização do negócio e objetivos
- Resultados detalhados (Scikit-learn + GLM)
- Gráficos embedados (Matriz de Confusão, Curva ROC, etc.)
- Tabelas de coeficientes GLM com significância
- Dashboard visual com distribuição de risco
- Recomendações acionáveis

📉 <span style="color: mediumseagreen;"><b> SEÇÃO 8: DASHBOARD EXCEL EXECUTIVO </b></span></p>
- Múltiplas abas temáticas
- Formatação condicional por risco
- Gráficos dinâmicos para acompanhamento
- KPIs resumidos para a gestão

In [1]:
# ==============================================================================
# 0. IMPORTAÇÕES, CONFIGURAÇÕES E FUNÇÕES AUXILIARES
# ==============================================================================

# --- 0.1. Importações ---
import pandas as pd
import numpy as np
import statsmodels.api as sm
import logging
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline # Importe o Pipeline padrão
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve, precision_recall_curve
)
from sklearn.feature_selection import VarianceThreshold
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
import os
import base64
import gc  # Para gerenciamento de memória
import shutil
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.genmod.generalized_linear_model import GLM
from datetime import datetime

# Criar diretório logs
os.makedirs("logs", exist_ok=True)

# Criar logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

# Limpar handlers anteriores
logger.handlers.clear()

# Timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = f"logs/execucao_{timestamp}.log"

# Handler arquivo
file_handler = logging.FileHandler(log_file, encoding='utf-8')
file_handler.setLevel(logging.DEBUG)

# Handler console
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)

# Formato
formatter = logging.Formatter(
    '[%(asctime)s] [%(levelname)s] %(message)s',
    datefmt='%d/%m/%Y %H:%M:%S'
)
file_handler.setFormatter(formatter)
console_handler.setFormatter(formatter)

# Adicionar handlers
logger.addHandler(file_handler)
logger.addHandler(console_handler)

# ✓ Pronto para usar!
logger.info("✓ Logger configurado com sucesso!")

logger.info("✓ Bibliotecas importadas e logger configurado com sucesso.")

# --- 0.2. Definição de Constantes ---
PATH_ARQUIVO = r'X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx'
SHEET_NAME = 'HC'
OUTPUT_DIR = 'regressao_logistica_output_ul' # Diretório para salvar saídas
OUTPUT_EXCEL_PREVISOES_FILE = os.path.join(OUTPUT_DIR, 'df_previsoes_risco_ul.xlsx')
OUTPUT_HTML_FILE = os.path.join(OUTPUT_DIR, 'relatorio_regressao_logistica_ul.html')
OUTPUT_GLM_COEF_CSV = os.path.join(OUTPUT_DIR, 'glm_coeficientes_summary_ul.csv')
OUTPUT_GLM_SUMMARY_TXT = os.path.join(OUTPUT_DIR, 'relatorio_glm_statsmodels_ul.txt')
OUTPUT_DASHBOARD_EXCEL_FILE = os.path.join(OUTPUT_DIR, 'dashboard_gestao_pessoas_ul.xlsx')

# Carregamento da base de controle de processos
id = 28
path_registros_processos = r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'
registros_processos = pd.read_excel(path_registros_processos, sheet_name = "REGISTROS", engine='openpyxl')
wb_p = load_workbook(path_registros_processos)
ws_p = wb_p['REGISTROS']
tempo_0 = [id, datetime.today(), 0]
ws_p.append(tempo_0)
wb_p.save(path_registros_processos)

# Criar diretório de saída se não existir
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    logger.info(f"✓ Diretório de saída '{OUTPUT_DIR}' criado.")

# --- 0.3. Funções Auxiliares para HTML ---
def converter_grafico_para_base64(filepath):
    """Converte um arquivo de imagem PNG para uma string Base64."""
    if not os.path.exists(filepath):
        logger.warning(f"Arquivo de gráfico não encontrado: {filepath}. Retornando string vazia.")
        return ""
    try:
        with open(filepath, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
        return f"data:image/png;base64,{encoded_string}"
    except Exception as e:
        logger.error(f"Erro ao converter gráfico {filepath} para Base64: {e}")
        return ""

def gerar_tabela_html(df, title="", caption="", limit=None):
    """Gera uma tabela HTML a partir de um DataFrame Pandas."""
    if df is None or df.empty:
        return f"<p><strong>{title}</strong>: Dados não disponíveis ou DataFrame vazio.</p>"
    
    # Limitar o número de linhas para exibição no HTML se especificado
    if limit and len(df) > limit:
        df_display = df.head(limit).copy()
        caption_text = f"{caption} (Mostrando as primeiras {limit} linhas de {len(df)})"
    else:
        df_display = df.copy()
        caption_text = caption

    html = f"""
    <div class="table-responsive">
        <h4 class="mt-4">{title}</h4>
        <p class="text-muted">{caption_text}</p>
        {df_display.to_html(classes="table table-striped table-bordered table-hover", index=False if 'Feature' in df.columns else True)}
    </div>
    """
    return html

def gerar_card_metricas(title, value, description=""):
    """Gera um card HTML para exibição de métricas."""
    return f"""
    <div class="col-md-4 mb-4">
        <div class="card shadow-sm border-0">
            <div class="card-body">
                <h5 class="card-title text-primary">{title}</h5>
                <h3 class="card-text text-dark">{value}</h3>
                <p class="card-text text-muted">{description}</p>
            </div>
        </div>
    </div>
    """

logger.info("✓ Funções auxiliares HTML configuradas.")

# ==============================================================================
# 1. CARREGAMENTO E PRÉ-PROCESSAMENTO INICIAL DOS DADOS
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 1 - CARREGAMENTO E PRÉ-PROCESSAMENTO INICIAL DOS DADOS")
logger.info("="*80)

try:
    # 1.1. Carregando a base de dados
    if not os.path.exists(PATH_ARQUIVO):
        raise FileNotFoundError(f"Arquivo não encontrado: {PATH_ARQUIVO}")

    dados_hc = pd.read_excel(PATH_ARQUIVO, sheet_name=SHEET_NAME, engine='openpyxl')
    logger.info(f"✓ Base de dados carregada com {dados_hc.shape[0]} registros e {dados_hc.shape[1]} colunas.")

    # Armazenar o DataFrame original para exportação posterior com todas as colunas
    df_hc_original_full = dados_hc.copy()

    # 1.2. Tratamento inicial de datas para filtragem
    # Garante que 'ano_rescisao' e 'ano_atestado' são tratados como strings para comparação
    # Preenche NaN com string vazia antes de converter para string
    dados_hc['ano_rescisao'] = dados_hc['ano_rescisao'].fillna('').astype(str)
    dados_hc['ano_atestado'] = dados_hc['ano_atestado'].fillna('').astype(str)
    logger.info("✓ Coluna 'ano_rescisao' tratada para string.")
    logger.info("✓ Coluna 'ano_atestado' tratada para string.")

    # 1.3. Aplicando filtros nos dados
    # A máscara filtra por ano de rescisão/atestado a partir de 2022 (ou sem data) e unidade 'UL'
    # Aqui removemos a variável'ano_atestado' para considerar todos colaboradores com ou sem
    mascara = (
        ((dados_hc['ano_rescisao'] >= '2022') | (dados_hc['ano_rescisao'] == '')) &
        (dados_hc['unidade'] == 'UL')
    )
    df_hc = dados_hc.loc[mascara].copy() # DataFrame filtrado para o modelo
    logger.info(f"✓ Dados filtrados: {df_hc.shape[0]} registros após aplicar critérios.")
    
    # Armazenar o DataFrame filtrado original (com todas as colunas) para exportação posterior
    df_hc_filtered_full = df_hc.copy()

    # 1.4. Excluindo variáveis que não serão utilizadas (Data Leakage e irrelevantes)
    # 'descricao_rescisao' é um exemplo CRÍTICO de data leakage se o objetivo é prever a demissão.
    colunas_para_dropar = [
        'registro','nome','nascimento','etnia_raca','cod_formacao','aposentado','rg','cpf','deficiente','nacionalidade',
        'naturalidade_uf','naturalidade_cidade','sexo_conjuge','nome_conjuge','data_nasc_conjuge','cpf_conjuge','data_admissao',
        'situacao','data_rescisao','descricao_rescisao','cod_cargo','cargo_completo','cargo_abreviado','cod_centro_custo',
        'nome_gestor','cod_secao','secao','salario_admissao','salario_atual','saldo_fgts','endereco',
        'numero_endereco','complemento','bairro','cidade','estado','cep','telefone','email','segundo_email','cod_situacao','regime_trabalho',
        'hora_base','hora_complemento','ultimo_reajuste_individual','ultimo_reajuste_coletivo','data_inicio_ferias',
        'data_fim_ferias','cod_empresa','empresa','unidade','ano_admissao','mes_admissao','ano_rescisao','mes_rescisao',
        'cargo_padrao','prazo_experiencia','exp_45dias','exp_90dias','cota_pcd','trab_reabilitado','horas_nao_trabalhadas','data_inicio_atestado','ano_atestado'
    ]
    colunas_existentes_para_dropar = [col for col in colunas_para_dropar if col in df_hc.columns]
    hc_padr = df_hc.drop(columns=colunas_existentes_para_dropar).copy()

    logger.info(f"✓ Variáveis removidas: {len(colunas_existentes_para_dropar)}")
    logger.info(f"✓ Variáveis mantidas para análise: {hc_padr.shape[1]} -> {hc_padr.columns.tolist()}")

    # 1.5. Tratamento de valores faltantes e coluna 'filhos'
    # Converte 'S'/'N' para 1/0 e preenche faltantes com 0 (assumindo sem filhos se não especificado)
    if 'filhos' in hc_padr.columns:
        faltantes_filhos = hc_padr['filhos'].isnull().sum()
        hc_padr['filhos'] = hc_padr['filhos'].astype(str).str.upper()
        hc_padr['filhos'] = hc_padr['filhos'].replace({'S': 1, 'N': 0})
        hc_padr['filhos'] = pd.to_numeric(hc_padr['filhos'], errors='coerce')
        hc_padr['filhos'] = hc_padr['filhos'].fillna(0).astype(int)
        logger.info(f"✓ Valores faltantes em 'filhos' tratados: {faltantes_filhos} registros preenchidos com 0 e convertidos para int.")
    else:
        logger.warning("Aviso: Coluna 'filhos' não encontrada. Verifique se foi removida acidentalmente.")

    # Após o bloco onde 'filhos' é convertido para int
    if 'filhos' in hc_padr.columns:
        hc_padr['filhos'] = hc_padr['filhos'].astype('category')

    # 1.6. Criando a variável alvo (target)
    # 'status' é convertido para 'target' (1 para INATIVO, 0 para ATIVO)
    if 'status' in hc_padr.columns:
        hc_padr['target'] = hc_padr['status'].apply(lambda x: 1 if x == 'INATIVO' else 0)
        logger.info("✓ Variável 'target' criada: INATIVO = 1, ATIVO = 0.")
        logger.info(f"Distribuição do target: \n{hc_padr['target'].value_counts()}")
    else:
        raise ValueError("Coluna 'status' não encontrada em hc_padr. Não é possível criar a variável target.")

    # 1.7. Validação de dados após pré-processamento
    logger.info(f"Número final de registros para modelagem: {hc_padr.shape[0]}")
    logger.info(f"Número final de colunas para modelagem: {hc_padr.shape[1]}")
    if hc_padr.isnull().sum().sum() == 0:
        logger.info("✓ Nenhum valor nulo restante no DataFrame para modelagem.")
    else:
        logger.warning(f"Aviso: Ainda existem valores nulos no DataFrame para modelagem: {hc_padr.isnull().sum()[hc_padr.isnull().sum() > 0].to_dict()}")

    logger.info("\nPRÉ-PROCESSAMENTO DOS DADOS CONCLUÍDO COM SUCESSO!")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 1 (Carregamento e Pré-processamento Inicial): {e}")
    sys.exit("Script encerrado devido ao erro no pré-processamento inicial.")

gc.collect() # Limpeza de memória
logger.info("✓ GC.collect() executado após Seção 1.")

logger.info("Aplicando agrupamento de variáveis categóricas (Estado Civil, Filhos, Formação)...")

    # -------------------------------------------------------------------------
    # 1. AGRUPAMENTO: ESTADO CIVIL
    # Regra: Manter S, C, D. Todo o resto vira 'OUTROS'.
    # -------------------------------------------------------------------------
def agrupar_estado_civil(valor):
    valor = str(valor).upper().strip()
    if valor in ['S', 'C', 'D']:
        return valor
    else:
        return 'OUTROS'

hc_padr['estado_civil'] = hc_padr['estado_civil'].apply(agrupar_estado_civil)
logger.info(f"Estado Civil agrupado. Categorias finais: {hc_padr['estado_civil'].unique()}")

    # -------------------------------------------------------------------------
    # 2. AGRUPAMENTO: FILHOS
    # Regra: Se for 'S' mantém 'S', qualquer outra coisa (N, vazio, nan) vira 'N'.
    # -------------------------------------------------------------------------
def agrupar_filhos(valor):
    valor = str(valor).upper().strip()
    if valor == 'S':
        return 'S'
    else:
        return 'N'

hc_padr['filhos'] = hc_padr['filhos'].apply(agrupar_filhos)
logger.info(f"Filhos agrupado. Categorias finais: {hc_padr['filhos'].unique()}")

    # -------------------------------------------------------------------------
    # 3. AGRUPAMENTO: FORMAÇÕES
    # Regra: Agrupar níveis de escolaridade conforme solicitado.
    # -------------------------------------------------------------------------
def agrupar_formacao(valor):
    v = str(valor).upper().strip()
        
    # Mapeamento de POS-GRADUADO
    if v in ['POS-GRADUADO', 'MESTRADO', 'DOUTORADO']:
        return 'POS-GRADUADO'
        
    # Mapeamento de ENSINO SUPERIOR
    elif 'SUPERIOR' in v: # Pega COMPLETO e INCOMPLETO
        return 'EDUCACAO_SUPERIOR'
        
    # Mapeamento de ENSINO MEDIO
    elif 'MEDIO' in v: # Pega COMPLETO e INCOMPLETO
        return 'ENSINO_MEDIO'
        
    # Mapeamento de ENSINO FUNDAMENTAL (Inclui 5o ano, até 5o ano, etc)
    elif 'FUNDAMENTAL' in v or '5O ANO' in v or 'ATE 5O' in v:
        return 'ENSINO_FUNDAMENTAL'
        
    # Mapeamento de ANALFABETO
    elif 'ANALFABETO' in v:
        return 'ANALFABETO'
        
    # Qualquer outra coisa (ex: IGNORADO, NÃO INFORMADO)
    else:
        return 'OUTROS'

hc_padr['formacoes'] = hc_padr['formacoes'].apply(agrupar_formacao)
logger.info(f"Formações agrupadas. Categorias finais: {hc_padr['formacoes'].unique()}")

# ==============================================================================
# 2. SEPARAÇÃO DE FEATURES (X) E TARGET (y)
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 2 - SEPARAÇÃO DE FEATURES (X) E TARGET (y)")
logger.info("="*80)

try:
    # CRÍTICO: Remover 'status' (coluna original) e 'target' (a própria variável alvo) de X
    # As colunas 'nome', 'cargo', etc. também devem ser removidas de X, mas mantidas no df_hc_filtered_full
    # para serem re-adicionadas posteriormente para o dashboard.
    colunas_para_dropar_final = ['target', 'status']
    # Certifique-se de que colunas como 'nome', 'cargo', etc. não estejam em X, mas sim no df_hc_filtered_full
    # X deve conter apenas as variáveis preditoras (numéricas e categóricas).
    
    # Identificar colunas não-numéricas que devem ser tratadas como categóricas ou que são identificadores e devem ser removidas.
    cols_to_exclude_from_X = ['status'] # 'target' já foi excluído acima
    
    # As colunas 'nome', 'cargo', etc. são úteis no df_hc_filtered_full para o relatório final,
    # mas não devem ser features no modelo.
    # Assumindo que hc_padr já tem as colunas relevantes e removeu as demais.
    
    X = hc_padr.drop(columns=[col for col in colunas_para_dropar_final if col in hc_padr.columns]).copy()
    y = hc_padr['target'].copy()

    logger.info("✓ Features (X) e Target (y) separados.")
    logger.info(f"✓ Features (X) selecionadas ({X.shape[1]} variáveis):")
    for i, col in enumerate(X.columns):
        logger.info(f"   {i+1}. {col}")
    logger.info(f"\n✓ Target (y) selecionado: {y.name}")
    logger.info(f"✓ Dimensões resultantes: X ({X.shape[0]}x{X.shape[1]}), y ({y.shape[0]})")

    logger.info("\n✓ SEÇÃO 2 CONCLUÍDA COM SUCESSO")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 2 (Separação de Features e Target): {e}")
    sys.exit("Script encerrado devido ao erro na separação de features/target.")

gc.collect() # Limpeza de memória
logger.info("✓ GC.collect() executado após Seção 2.")

# ==============================================================================
# 3. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 3 - ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)")
logger.info("="*80)

try:
    # 3.1. Estatísticas Descritivas das Variáveis Numéricas
    numerical_cols = X.select_dtypes(include=np.number).columns.tolist()
    logger.info("\n--- 3.1. Estatísticas Descritivas das Variáveis Numéricas ---")
    eda_desc_stats_df = X[numerical_cols].describe().T # DataFrame para HTML
    logger.info(eda_desc_stats_df)

    # 3.2. Distribuição da Variável Alvo ('target')
    logger.info("\n--- 3.2. Distribuição da Variável Alvo ('target') ---")
    eda_target_dist_df = y.value_counts().to_frame(name='count') # DataFrame para HTML
    prop_inativos = y.value_counts(normalize=True).get(1, 0) * 100
    prop_ativos = y.value_counts(normalize=True).get(0, 0) * 100
    logger.info(f"Proporção de 'INATIVOS' (1): {prop_inativos:.2f}%")
    logger.info(f"Proporção de 'ATIVOS' (0): {prop_ativos:.2f}%")

    plt.figure(figsize=(6, 4))
    sns.countplot(x=y) # Usar o 'y' diretamente
    plt.title('Distribuição da Variável Alvo (0=Ativo, 1=Inativo)')
    plt.xlabel('Status')
    plt.ylabel('Contagem')
    plt.xticks(ticks=[0, 1], labels=['ATIVO', 'INATIVO'])
    eda_target_dist_path = os.path.join(OUTPUT_DIR, 'distribuicao_target.png')
    plt.savefig(eda_target_dist_path)
    plt.close() # Fechar o plot para liberar memória
    eda_target_dist_base64 = converter_grafico_para_base64(eda_target_dist_path)
    logger.info(f"✓ Gráfico de distribuição do target salvo e convertido.")

    # 3.3. Análise de Correlação (Variáveis Numéricas)
    logger.info("\n--- 3.3. Análise de Correlação (Variáveis Numéricas) ---")
    # Incluir 'target' para calcular correlação
    correlation_matrix = pd.concat([X[numerical_cols], y], axis=1).corr()
    eda_corr_matrix_df = correlation_matrix # DataFrame para HTML
    logger.info("Matriz de Correlação:")
    logger.info(correlation_matrix)
    logger.info("\nCorrelação das variáveis numéricas com 'target':")
    eda_corr_target_df = correlation_matrix['target'].sort_values(ascending=False).to_frame(name='Correlação com Target')
    logger.info(eda_corr_target_df)

    plt.figure(figsize=(10, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Heatmap da Matriz de Correlação')
    eda_corr_heatmap_path = os.path.join(OUTPUT_DIR, 'heatmap_correlacao.png')
    plt.savefig(eda_corr_heatmap_path)
    plt.close()
    eda_corr_heatmap_base64 = converter_grafico_para_base64(eda_corr_heatmap_path)
    logger.info(f"✓ Heatmap da matriz de correlação salvo e convertido.")

    # 3.4. Visualizações Detalhadas das Distribuições
    logger.info("\n--- 3.4. Visualizações Detalhadas das Distribuições ---")
    
    try:
        # =========================================================================
        # PARTE 1: HEATMAP AMPLIADO (Todas as variáveis numéricas)
        # =========================================================================
        logger.info("\n  Gerando Heatmap de Correlação (ampliado)...")
        
        # Incluir 'target' para calcular correlação
        correlation_matrix = pd.concat([X[numerical_cols], y], axis=1).corr()
        
        plt.figure(figsize=(16, 14))
        sns.heatmap(
            correlation_matrix,
            annot=True,
            cmap='coolwarm',
            fmt=".2f",
            square=True,
            cbar_kws={'label': 'Correlação'},
            linewidths=0.5,
            linecolor='gray',
            vmin=-1,
            vmax=1,
            annot_kws={'size': 10}
        )
        plt.title('Heatmap da Matriz de Correlação (Todas as Variáveis Numéricas)', 
                  fontsize=16, fontweight='bold', pad=20)
        plt.xticks(rotation=45, ha='right', fontsize=10)
        plt.yticks(rotation=0, fontsize=10)
        plt.tight_layout()
        
        eda_corr_heatmap_path = os.path.join(OUTPUT_DIR, 'heatmap_correlacao_ampliado.png')
        plt.savefig(eda_corr_heatmap_path, dpi=300, bbox_inches='tight')
        plt.close()
        eda_corr_heatmap_base64 = converter_grafico_para_base64(eda_corr_heatmap_path)
        logger.info(f"  ✓ Heatmap ampliado salvo: {eda_corr_heatmap_path}")
        
        # =========================================================================
        # PARTE 2: HISTOGRAMAS E BOXPLOTS (Todas as variáveis numéricas)
        # =========================================================================
        logger.info("\n  Gerando Histogramas e Boxplots para variáveis numéricas...")
        
        eda_hist_box_base64 = {}
        
        # Iterar sobre TODAS as variáveis numéricas (não limitar a 3)
        for col in numerical_cols:
            try:
                plt.figure(figsize=(14, 5))
                
                # Subplot 1: Histograma com KDE
                plt.subplot(1, 2, 1)
                sns.histplot(X[col], kde=True, bins=30, color='steelblue', edgecolor='black')
                plt.title(f'Histograma de {col}', fontsize=12, fontweight='bold')
                plt.xlabel(col, fontsize=11)
                plt.ylabel('Frequência', fontsize=11)
                plt.grid(True, alpha=0.3)
                
                # Subplot 2: Boxplot
                plt.subplot(1, 2, 2)
                sns.boxplot(x=X[col], color='lightcoral', width=0.5)
                plt.title(f'Boxplot de {col}', fontsize=12, fontweight='bold')
                plt.xlabel(col, fontsize=11)
                plt.grid(True, alpha=0.3, axis='y')
                
                plt.tight_layout()
                filepath = os.path.join(OUTPUT_DIR, f'distribuicao_{col}.png')
                plt.savefig(filepath, dpi=300, bbox_inches='tight')
                plt.close()
                
                eda_hist_box_base64[col] = converter_grafico_para_base64(filepath)
                logger.info(f"    ✓ {col}")
                
            except Exception as e:
                logger.warning(f"    ✗ Erro ao gerar gráfico para {col}: {e}")
                continue
        
        logger.info(f"  ✓ Total de histogramas/boxplots gerados: {len(eda_hist_box_base64)}")
        
        # =========================================================================
        # PARTE 3: COUNTPLOTS (Todas as variáveis categóricas)
        # =========================================================================
        logger.info("\n  Gerando Countplots para variáveis categóricas...")
        
        eda_cat_target_base64 = {}
        categorical_cols = X.select_dtypes(include='object').columns.tolist()
        
        # Iterar sobre TODAS as variáveis categóricas (não limitar a 3)
        for col in categorical_cols:
            try:
                # Calcular o número de categorias para ajustar figsize dinamicamente
                n_categories = X[col].nunique()
                figsize_width = max(10, min(16, n_categories * 1.5))
                
                plt.figure(figsize=(figsize_width, 6))
                
                sns.countplot(
                    data=X,
                    x=col,
                    hue=y,
                    palette='Set2',
                    edgecolor='black',
                    linewidth=0.5
                )
                
                plt.title(f'Distribuição de {col} por Target (Ativo/Inativo)', 
                         fontsize=12, fontweight='bold', pad=15)
                plt.xlabel(col, fontsize=11, fontweight='bold')
                plt.ylabel('Contagem', fontsize=11, fontweight='bold')
                plt.xticks(rotation=45, ha='right', fontsize=10)
                plt.legend(title='Target', labels=['Ativo (0)', 'Inativo (1)'], 
                          fontsize=10, title_fontsize=11)
                plt.grid(True, alpha=0.3, axis='y')
                plt.tight_layout()
                
                filepath = os.path.join(OUTPUT_DIR, f'distribuicao_categorica_{col}.png')
                plt.savefig(filepath, dpi=300, bbox_inches='tight')
                plt.close()
                
                eda_cat_target_base64[col] = converter_grafico_para_base64(filepath)
                logger.info(f"    ✓ {col} ({n_categories} categorias)")
                
            except Exception as e:
                logger.warning(f"    ✗ Erro ao gerar gráfico para {col}: {e}")
                continue
        
        logger.info(f"  ✓ Total de countplots gerados: {len(eda_cat_target_base64)}")
        
        # =========================================================================
        # PARTE 4: RESUMO FINAL
        # =========================================================================
        logger.info("\n--- RESUMO DA SEÇÃO 3.4 ---")
        logger.info(f"✓ Heatmap de correlação: 1 arquivo")
        logger.info(f"✓ Histogramas/Boxplots: {len(eda_hist_box_base64)} arquivos")
        logger.info(f"✓ Countplots categóricos: {len(eda_cat_target_base64)} arquivos")
        logger.info(f"✓ Total de gráficos gerados: {1 + len(eda_hist_box_base64) + len(eda_cat_target_base64)}")
        logger.info("\n✓ SEÇÃO 3.4 CONCLUÍDA COM SUCESSO!")
        
    except Exception as e:
        logger.error(f"ERRO na SEÇÃO 3.4 (Visualizações): {e}")
        import traceback
        traceback.print_exc()
        sys.exit("Script encerrado devido ao erro na Seção 3.4.")

    gc.collect()
    logger.info("✓ GC.collect() executado após Seção 3.4.")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 3 (Análise Exploratória dos Dados): {e}")
    sys.exit("Script encerrado devido ao erro na EDA.")

gc.collect() # Limpeza de memória
logger.info("✓ GC.collect() executado após Seção 3.")

# ==============================================================================
# 4. CONSTRUÇÃO E AVALIAÇÃO DO MODELO (Scikit-learn e Statsmodels)
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 4 - CONSTRUÇÃO E AVALIAÇÃO DO MODELO")
logger.info("="*80)

# Variáveis para armazenar resultados do modelo Scikit-learn
smote_pipeline = None
optimal_threshold = 0.5
accuracy_skl, precision_skl, recall_skl, f1_skl, roc_auc_skl, gini = [0.0]*6
cv_scores_mean, cv_scores_std = 0.0, 0.0
classification_report_str = ""
confusion_matrix_base64 = ""
roc_curve_base64 = ""

# Variáveis para armazenar resultados do modelo GLM
glm_results = None
glm_summary_df = pd.DataFrame()
glm_loglike = "N/A"
glm_aic = "N/A"
glm_bic = "N/A"

try:
    # ==============================================================================
    # 4.0. DEFINIÇÃO INICIAL DE FEATURES (NECESSÁRIO ANTES DE 4.1.A)
    # ==============================================================================
    logger.info("\n--- 4.0. Identificação de Features Numéricas e Categóricas ---")
    
    numerical_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(include='object').columns.tolist()
    
    logger.info(f"✓ Features Numéricas identificadas: {numerical_features}")
    logger.info(f"✓ Features Categóricas identificadas: {categorical_features}")
    
    # ==============================================================================
    # 4.1.A. SELEÇÃO AUTOMÁTICA DE FEATURES (PRÉ-PROCESSAMENTO)
    # ==============================================================================
    logger.info("\n" + "="*80)
    logger.info("SEÇÃO 4.1.A - SELEÇÃO AUTOMÁTICA DE FEATURES (VIF + CHI-SQUARE)")
    logger.info("="*80)

    try:
        from scipy.stats import chi2_contingency
        from sklearn.preprocessing import LabelEncoder
        from statsmodels.stats.outliers_influence import variance_inflation_factor
        
        # PASSO 1: Calcular p-values para features categóricas (Chi-Square)
        logger.info("\n--- PASSO 1: Análise de Associação (Categóricas) ---")
        
        cat_pvalues = {}
        for col in categorical_features:
            try:
                # Criar tabela de contingência
                contingency_table = pd.crosstab(X[col], y) # Criar tabela de contingência entre a feature categórica e o target (y)
                chi2, p_val, dof, expected = chi2_contingency(contingency_table) # Realizar o teste Chi-Square de independência
                cat_pvalues[col] = p_val
                logger.info(f"  {col}: χ² = {chi2:.2f}, p-value = {p_val:.4f}")
            except Exception as e:
                logger.warning(f"  {col}: Erro no Chi-Square - {e}. Considerar p-value = 1.0 (não significativo)")
                cat_pvalues[col] = 1.0 # Atribui p-value alto se houver erro, indicando não significância
        
        # =========================================================================
        # PASSO 2: Calcular p-values para features numéricas (ANOVA F-test)
        # =========================================================================
        logger.info("\n--- PASSO 2: Análise de Variância (Numéricas) ---")
        
        from sklearn.feature_selection import f_classif
        
        num_pvalues = {}
        if len(numerical_features) > 0:
            f_stats, f_pvals = f_classif(X[numerical_features], y) # Realizar o teste ANOVA F-test para cada feature numérica vs target (y)
            for col, p_val in zip(numerical_features, f_pvals):
                num_pvalues[col] = p_val
                logger.info(f"  {col}: F-stat p-value = {p_val:.4f}")
        
        # =========================================================================
        # PASSO 3: Consolidar e Filtrar por Significância (α = 0.05)
        # =========================================================================
        logger.info("\n--- PASSO 3: Consolidação de P-values ---")
        
        all_pvalues = {**num_pvalues, **cat_pvalues} # Combina os p-values de features numéricas e categóricas
        alpha = 0.05
        
        # Cria um DataFrame para visualizar os p-values
        pval_df = pd.DataFrame(list(all_pvalues.items()), columns=['Feature', 'p-value'])
        pval_df = pval_df.sort_values('p-value').reset_index(drop=True)
        
        logger.info("\nTodas as Features (ordenadas por p-value):")
        logger.info(pval_df.to_string(index=False))
        
        # Seleciona as features cujo p-value é menor que o nível de significância
        significant_features = pval_df[pval_df['p-value'] < alpha]['Feature'].tolist()
        removed_features = pval_df[pval_df['p-value'] >= alpha]['Feature'].tolist()
        
        logger.info(f"\n✓ Features SIGNIFICATIVAS (p < {alpha}): {len(significant_features)}")
        logger.info(f"  {significant_features}")
        logger.info(f"\n✗ Features REMOVIDAS (p ≥ {alpha}): {len(removed_features)}")
        logger.info(f"  {removed_features}")
        
        # =========================================================================
        # PASSO 4: Verificar Multicolinearidade (VIF) nas Numéricas
        # =========================================================================
        logger.info("\n--- PASSO 4: Análise de Multicolinearidade (VIF) ---")
        
        # Filtra apenas as features numéricas que foram consideradas significativas
        numerical_significant = [f for f in significant_features if f in numerical_features]
        
        if len(numerical_significant) > 1:
            from sklearn.preprocessing import StandardScaler
            X_num_scaled = StandardScaler().fit_transform(X[numerical_significant]) # Escala as features numéricas para o cálculo do VIF
            
            vif_results = []
            for i, col in enumerate(numerical_significant):
                try:
                    vif_val = variance_inflation_factor(X_num_scaled, i)
                    vif_results.append({'Feature': col, 'VIF': vif_val})
                    logger.info(f"  {col}: VIF = {vif_val:.2f}")
                except:
                    vif_results.append({'Feature': col, 'VIF': np.inf})
                    logger.warning(f"  {col}: VIF = ∞ (perfeita colinearidade)")
            
            vif_df = pd.DataFrame(vif_results).sort_values('VIF', ascending=False)
            
            # Remover features com VIF > 10 (limite comum para alta multicolinearidade)
            high_vif_features = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
            if high_vif_features:
                logger.warning(f"\n⚠️  Removendo por VIF > 10: {high_vif_features}") # Atualiza a lista de features significativas removendo as de alto VIF
                significant_features = [f for f in significant_features if f not in high_vif_features]
            else:
                logger.info("  Não há features numéricas suficientes para calcular VIF ou apenas uma.")
        
        # =========================================================================
        # PASSO 5: Atualizar Features e X
        # =========================================================================
        logger.info(f"\n--- PASSO 5: Consolidação Final ---")
        
        numerical_features_selected = [f for f in significant_features if f in numerical_features]
        categorical_features_selected = [f for f in significant_features if f in categorical_features]
        
        # Para o modelo GLM, 'empresa_resumo' é removida devido à alta multicolinearidade e separação perfeita que o Statsmodels não lida bem sem regularização explícita.
        categorical_features_selected_glm = [f for f in categorical_features_selected 
                                            if f != 'empresa_resumo']
        
         # Atualiza o DataFrame X para conter apenas as features selecionadas
        X = X[significant_features].copy()  # X para Scikit-learn (pode incluir 'empresa_resumo')
        
        # Atualiza as listas globais de features para serem usadas nas próximas etapas
        numerical_features = numerical_features_selected
        categorical_features = categorical_features_selected
        
        logger.info(f"\n✓ Features finais para Scikit-learn: {X.shape[1]}")
        logger.info(f"  Numéricas: {numerical_features}")
        logger.info(f"  Categóricas: {categorical_features}")
        logger.info(f"\n✓ Features finais para GLM (sem empresa_resumo): {len(numerical_features_selected) + len(categorical_features_selected_glm)}")
        logger.info(f"  Numéricas: {numerical_features_selected}")
        logger.info(f"  Categóricas: {categorical_features_selected_glm}")
        
        logger.info("\n✓ SEÇÃO 4.1.A CONCLUÍDA\n")

    except Exception as e:
        logger.error(f"ERRO na SEÇÃO 4.1.A: {e}")
        import traceback
        traceback.print_exc()
        sys.exit("Script encerrado devido ao erro na seleção de features.")

    gc.collect()

    # ==============================================================================
    # 4.1. Dividindo dados em Treino e Teste
    # ==============================================================================
    
    # Divide o dataset em conjuntos de treino e teste (80/20), estratificando pelo target (y)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    logger.info("\n--- 4.1. Dividindo dados em Treino e Teste ---")
    logger.info(f"✓ Dados divididos: {X_train.shape[0]} amostras de treino, {X_test.shape[0]} amostras de teste.")
    logger.info(f"Proporção de 'INATIVOS' (1) no treino: {y_train.value_counts(normalize=True).get(1, 0)*100:.2f}%")
    logger.info(f"Proporção de 'INATIVOS' (1) no teste: {y_test.value_counts(normalize=True).get(1, 0)*100:.2f}%")

    # ==============================================================================
    # 4.2. Preparação de Features (ColumnTransformer para Scikit-learn)
    # ==============================================================================    

    # Cria um ColumnTransformer para aplicar transformações diferentes a colunas diferentes
    # 'num': aplica StandardScaler (normalização) às features numéricas<br/>
    # 'cat': aplica OneHotEncoder (cria dummies) às features categóricas, removendo a primeira para evitar multicolinearidade<br/>
    # 'remainder='passthrough'': mantém colunas não especificadas (útil se houver outras colunas que não são features)

    preprocessor_skl = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='if_binary'), categorical_features)
        ],
        remainder='passthrough' # Manter colunas não transformadas
    )
    
    logger.info("\n--- 4.2. Preparação de Features (ColumnTransformer para Scikit-learn) ---")
    logger.info(f"✓ Features já selecionadas em 4.1.A:")
    logger.info(f"  Numéricas: {numerical_features}")
    logger.info(f"  Categóricas: {categorical_features}")
    logger.info("✓ Pré-processador (ColumnTransformer) configurado com sucesso.")

    # ==============================================================================
    # 4.3. Construção do Pipeline (Regressão Logística com Class Weight)
    # ==============================================================================

    # Define o pipeline que encadeia o pré-processamento e o classificador
    # 'preprocessor': aplica as transformações definidas em preprocessor_skl<br/>
    # 'classifier': utiliza Regressão Logística<br/>
    #   - solver='lbfgs': algoritmo de otimização<br/>
    #   - random_state=42: para reprodutibilidade<br/>
    #   - class_weight='balanced': ajusta os pesos das classes para lidar com desbalanceamento<br/>
    #   - max_iter=2000: aumenta o número de iterações para garantir convergência<br/>
    #   - n_jobs=-1: utiliza todos os núcleos da CPU para treinamento<br/>

    logger.info("\n--- 4.3. Construindo Pipeline (Sem SMOTE, com Class Weight) ---")

    from sklearn.pipeline import Pipeline

    try:
        smote_pipeline = Pipeline([
            ('preprocessor', preprocessor_skl),
            ('classifier', LogisticRegression(
                solver='lbfgs', 
                random_state=42, 
                class_weight='balanced', # Lida com o desbalanceamento de classes
                max_iter=2000, 
                n_jobs=-1
            ))
        ])
        
        logger.info("✓ Pipeline construído (Regressão Logística com class_weight='balanced').")

    except Exception as e:
        logger.error(f"Erro ao construir o pipeline: {e}")
        smote_pipeline = None

    # 4.4. Treinamento do Modelo e Validação Cruzada (Scikit-learn)
    logger.info("\n--- 4.4. Treinamento do Modelo e Validação Cruzada (Scikit-learn) ---")
    logger.info("Iniciando o treinamento do pipeline (Scikit-learn)...")
    smote_pipeline.fit(X_train, y_train)
    logger.info("✓ Pipeline Scikit-learn treinado com sucesso!")

    # Validação Cruzada (ROC AUC como métrica de desempenho)
    # Usa StratifiedKFold para manter a proporção das classes em cada fold
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(smote_pipeline, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_scores_mean = cv_scores.mean()
    cv_scores_std = cv_scores.std()
    logger.info(f"✓ Validação cruzada (ROC AUC) concluída: {cv_scores_mean:.4f} (+/- {cv_scores_std:.4f})")
    logger.info(f"Resultados de cada fold: {[f'{score:.4f}' for score in cv_scores]}")

    # Avaliação no conjunto de teste
    y_proba_skl = smote_pipeline.predict_proba(X_test)[:, 1]

    # Ajuste do threshold para otimizar F1-Score (balancear Precision e Recall)
    # A curva Precision-Recall ajuda a encontrar o melhor ponto de corte
    precision, recall, thresholds = precision_recall_curve(y_test, y_proba_skl)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10) # Adiciona 1e-10 para evitar divisão por zero
    optimal_threshold_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_threshold_idx]
    
    # Aplica o threshold otimizado para obter as previsões binárias
    y_pred_skl = (y_proba_skl >= optimal_threshold).astype(int)

    # Calcula as métricas de avaliação
    accuracy_skl = accuracy_score(y_test, y_pred_skl)
    precision_skl = precision_score(y_test, y_pred_skl)
    recall_skl = recall_score(y_test, y_pred_skl)
    f1_skl = f1_score(y_test, y_pred_skl)
    roc_auc_skl = roc_auc_score(y_test, y_proba_skl)
    gini = 2 * roc_auc_skl - 1 # Coeficiente de Gini

    logger.info("\n--- Avaliação do Modelo (Scikit-learn Logistic Regression) ---")
    logger.info(f"Threshold de classificação ajustado: {optimal_threshold:.4f}")
    logger.info(f"Acurácia: {accuracy_skl:.4f}")
    logger.info(f"Precisão: {precision_skl:.4f}")
    logger.info(f"Recall: {recall_skl:.4f}")
    logger.info(f"F1-Score: {f1_skl:.4f}")
    logger.info(f"ROC AUC: {roc_auc_skl:.4f}")
    logger.info(f"Coeficiente de Gini: {gini:.4f}")

    classification_report_str = classification_report(y_test, y_pred_skl, output_dict=False)
    logger.info("\nRelatório de Classificação:\n" + classification_report_str)

    # Matriz de Confusão
    cm_skl = confusion_matrix(y_test, y_pred_skl)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm_skl, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Previsto Ativo (0)', 'Previsto Inativo (1)'],
                yticklabels=['Real Ativo (0)', 'Real Inativo (1)'])
    plt.title(f'Matriz de Confusão (Scikit-learn) - Threshold={optimal_threshold:.2f}')
    plt.ylabel('Valor Real')
    plt.xlabel('Valor Previsto')
    confusion_matrix_path = os.path.join(OUTPUT_DIR, 'matriz_confusao_sklearn.png')
    plt.savefig(confusion_matrix_path)
    plt.close() # Fechar o plot para liberar memória
    confusion_matrix_base64 = converter_grafico_para_base64(confusion_matrix_path)
    logger.info(f"✓ Matriz de Confusão salva e convertida.")

    # Curva ROC com Coeficiente de Gini
    fpr, tpr, thresholds_roc = roc_curve(y_test, y_proba_skl)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc_skl:.2f}, Gini = {gini:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Linha de Referência Aleatória')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Taxa de Falso Positivo (1 - Especificidade)')
    plt.ylabel('Taxa de Verdadeiro Positivo (Sensibilidade)')
    plt.title(f'Curva ROC para Previsão de Demissão (Gini = {gini:.2f})')
    plt.legend(loc="lower right")
    plt.grid(True)
    roc_curve_path = os.path.join(OUTPUT_DIR, 'curva_roc_gini.png')
    plt.savefig(roc_curve_path)
    plt.close() # Fechar o plot para liberar memória
    roc_curve_base64 = converter_grafico_para_base64(roc_curve_path)
    logger.info(f"✓ Curva ROC com Gini salva e convertida.")
    
except Exception as e:
    logger.error(f"ERRO na SEÇÃO 4.4 (Scikit-learn): {e}")
    import traceback
    traceback.print_exc()
    sys.exit("Script encerrado devido ao erro no treinamento do modelo Scikit-learn.")

gc.collect()

# ==============================================================================
# 4.5.B. Modelo GLM (Generalized Linear Model) com Statsmodels
# ==============================================================================
logger.info("\n--- 4.5.B. Modelo GLM (Generalized Linear Model) com Statsmodels ---")

try:
    # 1) Pré-processador dedicado ao GLM com baseline nas dummies (drop='first')
    # 'empresa_resumo' é removida para o GLM devido a problemas de multicolinearidade e separação perfeita que o Statsmodels não lida bem sem regularização explícita.
    categorical_features_filtered_glm = [col for col in categorical_features if col != 'empresa_resumo']

    logger.info(f"✓ Features categóricas filtradas para GLM (sem 'empresa_resumo'): {categorical_features_filtered_glm}")

    # Criar o preprocessador GLM
    # Utiliza StandardScaler para features numéricas e OneHotEncoder para categóricas.
    # 'drop='first'' evita a armadilha da variável dummy, e 'handle_unknown='ignore'' previne erros.
    preprocessor_glm = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_features),
            ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first'), categorical_features_filtered_glm) # Usa features categóricas filtradas para GLM
        ],
        remainder='drop' # Descarta colunas não especificadas
    )
    logger.info(f"✓ Preprocessador GLM configurado com {len(categorical_features_filtered_glm)} features categóricas.")

    # 2)  Definir a função consolidar_categorias_raras
    # Esta função agrupa categorias com frequência abaixo de um limiar em 'OUTROS',
    # prevenindo problemas de separação perfeita e melhorando a estabilidade do modelo.
    def consolidar_categorias_raras(df, cols, min_freq=0.05): # min_freq ajustado para 5%
        df = df.copy()
        for c in cols:
            if c in df.columns and df[c].dtype == 'object':
                freq = df[c].value_counts(normalize=True)
                raras = freq.index[freq < min_freq]
                if len(raras) > 0:
                    logger.info(f"  Consolidando {len(raras)} categorias raras em '{c}' para 'OUTROS'.")
                    df[c] = df[c].where(~df[c].isin(raras), 'OUTROS')
        return df

    # Aplica a consolidação de categorias raras no conjunto de treino
    # Garante que o X_train_glm_raw já esteja com as categorias raras tratadas antes da transformação.
    X_train_glm_raw = consolidar_categorias_raras(X_train, categorical_features_filtered_glm, min_freq=0.05)
    logger.info(f"Categorias após consolidação: {X_train_glm_raw.nunique().to_dict()}")

    # 3) Transformar o conjunto de treino e reconstruir o DataFrame
    # Aplica as transformações (escalonamento e one-hot encoding) e recria um DataFrame com os nomes corretos das features para facilitar a interpretação.
    X_train_glm_arr = preprocessor_glm.fit_transform(X_train_glm_raw)
    logger.info(f"✓ Dados de treino transformados para GLM: {X_train_glm_arr.shape}.")

    glm_cat_names = []
    if len(categorical_features) > 0:
        # Obtém os nomes das features categóricas transformadas
        glm_cat_names = list(preprocessor_glm.named_transformers_['cat'].get_feature_names_out(categorical_features_filtered_glm))
        logger.info(f"Features categóricas após One-Hot: {len(glm_cat_names)} variáveis")

    # Combina nomes das features numéricas e categóricas transformadas
    feature_names_glm = numerical_features + glm_cat_names
    X_train_glm_df = pd.DataFrame(X_train_glm_arr, columns=feature_names_glm, index=X_train.index)
    logger.info(f"✓ DataFrame GLM reconstruído com {X_train_glm_df.shape[1]} features.")

    # 4) Remover colinearidade via VIF (Variance Inflation Factor) e detectar separação perfeita
    # Esta etapa é crucial para garantir a estabilidade e interpretabilidade dos coeficientes do GLM.
    # A função 'remover_vif_alto' elimina features com alta multicolinearidade.
    # A detecção de separação perfeita remove features que causam coeficientes infinitos.
    def remover_vif_alto(df, thresh=5.0, max_iter=20):
        df_work = df.copy()
        iteration = 0
        while iteration < max_iter:
            if df_work.shape[1] <= 1:
                logger.info(f"  ⚠️  Atingido limite de colunas (1) durante remoção VIF.")
                break
            
            vif_vals = []
            cols = df_work.columns.tolist()
            
            for i in range(df_work.shape[1]):
                try:
                    vif = variance_inflation_factor(df_work.values, i)
                    vif_vals.append(vif)
                except Exception as e:
                    logger.warning(f" Erro ao calcular VIF para coluna '{cols[i]}': {e}. Atribuindo VIF infinito.")
                    vif_vals.append(np.inf) # Atribui VIF infinito em caso de erro (colinearidade perfeita)
            
            vif_series = pd.Series(vif_vals, index=cols)
            col_max = vif_series.idxmax()
            vif_max = vif_series[col_max]
            
            if vif_max > thresh:
                logger.info(f" Removendo '{col_max}' (VIF={vif_max:.2f} > {thresh}).")
                df_work = df_work.drop(columns=[col_max])
                iteration += 1
            else:
                logger.info(f" ✓ Todos VIF < {thresh}. VIF máximo: {vif_max:.2f}.")
                break
        
        return df_work

    # Aplica a remoção de VIF
    X_train_glm_cleaned = remover_vif_alto(X_train_glm_df, thresh=5.0)
    logger.info(f"✓ Variáveis após limpeza VIF: {X_train_glm_cleaned.shape[1]}")

     # 5) Detectar e remover separação perfeita (CRUCIAL para evitar Odds Ratio infinito)
     # A separação perfeita ocorre quando uma feature prediz perfeitamente o outcome para um subgrupo
    sep_cols = []
    for col in X_train_glm_cleaned.columns:
        x_col = X_train_glm_cleaned[col]
        # Separação perfeita: todas as 1s em um grupo e todas as 0s em outro
        if (x_col[y_train == 1].sum() == 0 and x_col[y_train == 0].sum() > 0) or \
           (x_col[y_train == 0].sum() == 0 and x_col[y_train == 1].sum() > 0):
            sep_cols.append(col)

    X_train_glm_cleaned_sep = X_train_glm_cleaned.copy()
    if len(sep_cols) > 0:
        logger.warning(f"⚠️  Removendo {len(sep_cols)} colunas com separação perfeita: {sep_cols}.")
        X_train_glm_cleaned_sep = X_train_glm_cleaned.drop(columns=sep_cols)
    else:
        logger.info("✓ Nenhuma separação perfeita detectada nas features do GLM.")

    # 6) Adicionar intercepto aos DADOS ORIGINAIS LIMPOS
    X_train_glm_final = sm.add_constant(X_train_glm_cleaned_sep, prepend=True)
    logger.info(f"✓ Features GLM finais (com intercepto): {X_train_glm_final.shape[1]} variáveis (com intercepto).")

    # 7) Verificação de Features Removidas
    # Este passo fornece um log detalhado sobre quais features foram removidas em cada etapa
    # do pré-processamento do GLM, ajudando a entender o modelo final.
    logger.info("7: VERIFICAÇÃO DETALHADA DE FEATURES REMOVIDAS ---")

    # Features originais após OneHotEncoding
    original_glm_features = set(X_train_glm_df.columns)
    logger.info(f"  Total de features após OneHotEncoding: {len(original_glm_features)}")

    # Features após remoção por VIF
    features_after_vif = set(X_train_glm_cleaned.columns)
    removed_by_vif = original_glm_features - features_after_vif
    if removed_by_vif:
        logger.warning(f"  ⚠️  Features removidas por VIF alto (>5.0): {sorted(list(removed_by_vif))}")
    else:
        logger.info("  ✓ Nenhuma feature removida por VIF alto.")

    # Features após remoção por separação perfeita
    features_after_sep = set(X_train_glm_cleaned_sep.columns)
    removed_by_sep = features_after_vif - features_after_sep
    if removed_by_sep:
        logger.warning(f"  ⚠️  Features removidas por separação perfeita: {sorted(list(removed_by_sep))}")
    else:
        logger.info("  ✓ Nenhuma feature removida por separação perfeita.")

    # Verificação específica para 'formacoes'
    formacoes_original_cols = [col for col in original_glm_features if 'formacoes' in col]
    formacoes_final_cols = [col for col in X_train_glm_final.columns if 'formacoes' in col]

    if formacoes_original_cols:
        logger.info(f"  Categorias de 'formacoes' inicialmente presentes: {sorted(formacoes_original_cols)}")
        if not formacoes_final_cols:
            logger.critical(f"  🔴 TODAS as categorias de 'formacoes' foram removidas do modelo final.")
        else:
            logger.info(f"  Categorias de 'formacoes' no modelo final: {sorted(formacoes_final_cols)}")
            removed_formacoes = set(formacoes_original_cols) - set(formacoes_final_cols)
            if removed_formacoes:
                logger.warning(f"  ⚠️  Categorias de 'formacoes' removidas: {sorted(list(removed_formacoes))}")
            else:
                logger.info("  ✓ Nenhuma categoria de 'formacoes' foi removida.")
    else:
        logger.info("  ✓ Nenhuma categoria de 'formacoes' estava presente após OneHotEncoding.")

    logger.info("--- FIM DA VERIFICAÇÃO DE FEATURES REMOVIDAS ---")

    # 8) Treinar GLM
    # O modelo GLM é treinado com a família Binomial (para regressão logística) e os dados já limpos e preparados.
    logger.info("Treinando GLM (Binomial, Logit) com dados de treino...")
    glm_model = sm.GLM(y_train, X_train_glm_final, family=sm.families.Binomial())
    glm_results = glm_model.fit(disp=0) # disp=0 para evitar warnings de dispersão
    
    logger.info("✓ GLM treinado com sucesso!")

    # 9) Extrair métricas e coeficientes do GLM
    # Calcula e loga métricas de ajuste do modelo (Log-Likelihood, AIC, BIC) e extrai os coeficientes, erros padrão, p-values e Odds Ratios para análise.
    glm_loglike = glm_results.llf
    glm_aic = glm_results.aic
    glm_bic = glm_results.bic
    
    logger.info(f"  Log-Likelihood: {glm_loglike:.2f}")
    logger.info(f"  AIC: {glm_aic:.2f}")
    logger.info(f"  BIC: {glm_bic:.2f}")

    betas = glm_results.params
    odds_ratio = np.exp(betas.values)

    glm_summary_df = pd.DataFrame({
        'Feature': betas.index,
        'Coefficient (Beta)': betas.values,
        'Std Err': glm_results.bse.values,
        'p-value': glm_results.pvalues.values,
        'Odds Ratio': odds_ratio,
        'Significância': ['***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '' 
                         for p in glm_results.pvalues.values]
    }).sort_values(by='p-value').reset_index(drop=True)

    logger.info(f"\nTOP 10 Features Significativas do GLM:")
    logger.info(glm_summary_df[glm_summary_df['Significância'] != ''].head(10).to_string())

    # Exporta os resultados do GLM para arquivos CSV e TXT
    OUTPUT_GLM_COEF_CSV = os.path.join(OUTPUT_DIR, 'glm_coefficients.csv')
    OUTPUT_GLM_SUMMARY_TXT = os.path.join(OUTPUT_DIR, 'glm_summary.txt')
    glm_summary_df.to_csv(OUTPUT_GLM_COEF_CSV, index=False)
    
    with open(OUTPUT_GLM_SUMMARY_TXT, 'w') as f:
        f.write(glm_results.summary().as_text())

    logger.info(f"✓ Coeficientes GLM exportados para '{OUTPUT_GLM_COEF_CSV}'.")
    logger.info(f"✓ Sumário GLM exportado para '{OUTPUT_GLM_SUMMARY_TXT}'.")
    
except Exception as e:
    logger.error(f"ERRO CRÍTICO na SEÇÃO 4.5.B: {e}")
    import traceback
    traceback.print_exc()
    glm_results = None
    glm_summary_df = pd.DataFrame()
    sys.exit("Script encerrado devido ao erro na Seção 4.5.B (Modelo GLM).")

    gc.collect() # Limpeza de memória
    logger.info("✓ GC.collect() executado após Seção 4.5.B.")

    # ==============================================================================
    # 4.5.C. NOVO: Modelo com Regularização L2 (Sklearn) para Comparação
    # ==============================================================================
    logger.info("\n--- 4.5.C. Modelo com Regularização L2 (Sklearn) para Comparação ---")
    
    try:
        # Treinar Logistic Regression com L2 regularization (Ridge)
        # Este modelo complementa o GLM fornecendo uma versão com regularização explícita
        glm_sklearn = LogisticRegression(
            penalty='l2',                 # Ridge regularization
            C=1.0,                       # Inverse of regularization strength (smaller = more regularization)
            solver='lbfgs',              # Compatível com multi-class
            max_iter=1000,
            class_weight='balanced',     # Balanceia classes
            random_state=42,
            n_jobs=-1
        )
        
        # Usar os mesmos dados de X_train_glm_final (já preprocessados)
        glm_sklearn.fit(X_train_glm_final, y_train)
        
        logger.info("✓ Logistic Regression com L2 treinada com sucesso!")
        
        # Extrair coeficientes
        coef_l2 = glm_sklearn.coef_[0]
        intercept_l2 = glm_sklearn.intercept_[0]
        
        logger.info(f"  Intercepto (L2): {intercept_l2:.4f}")
        logger.info(f"  Coeficientes (primeiros 5): {coef_l2[:5]}")
        
        # Avaliar no conjunto de teste
        y_pred_glm_sklearn = glm_sklearn.predict(X_train_glm_final)
        y_proba_glm_sklearn = glm_sklearn.predict_proba(X_train_glm_final)[:, 1]
        
        # Calcular AUC (se houver dados de teste)
        auc_glm_sklearn = roc_auc_score(y_train, y_proba_glm_sklearn)
        logger.info(f"  AUC (treino, L2): {auc_glm_sklearn:.4f}")
        
        # Comparar coeficientes: Statsmodels vs Sklearn
        logger.info("\n--- Comparação de Coeficientes: Statsmodels GLM vs Sklearn L2 ---")
        
        # Statsmodels usa nomes de features, Sklearn usa índices
        feature_names_from_glm = glm_summary_df['Feature'].values
        
        comparison_df = pd.DataFrame({
            'Feature': feature_names_from_glm,
            'Statsmodels_Beta': betas.values,
            'Sklearn_Coef_L2': coef_l2  # Note: primeiro coef de Sklearn corresponde à primeira feature não-intercepto
        })
        
        logger.info(comparison_df.head(10).to_string())
        
        # Exportar comparação
        COMPARISON_CSV = os.path.join(OUTPUT_DIR, 'glm_comparison_statsmodels_vs_sklearn_l2.csv')
        comparison_df.to_csv(COMPARISON_CSV, index=False)
        logger.info(f"✓ Comparação de coeficientes exportada para '{COMPARISON_CSV}'.")
        
        logger.info("✓ SEÇÃO 4.5.C CONCLUÍDA\n")
        
    except Exception as e:
        logger.error(f"ERRO na SEÇÃO 4.5.C (Sklearn L2): {e}")
        import traceback
        traceback.print_exc()
        logger.warning("⚠️  Continuando sem o modelo Sklearn L2...")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 4.5.B: {e}")
    import traceback
    traceback.print_exc()
    glm_results = None
    glm_summary_df = pd.DataFrame()

gc.collect()

# ==============================================================================
# 5. PREVISÕES E EXPORTAÇÃO MELHORADA DOS RESULTADOS
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 5 - PREVISÕES E EXPORTAÇÃO MELHORADA DOS RESULTADOS")
logger.info("="*80)

df_resultados = pd.DataFrame()  # Inicializar para garantir que existe

try:
    if smote_pipeline is None:
        raise ValueError("Pipeline Scikit-learn não foi treinado. Não é possível fazer previsões.")

    # 5.1. Calculando probabilidades e previsões para o DataFrame original completo (filtrado)
    prob_demissao_completa = smote_pipeline.predict_proba(X)[:, 1]  # Previsão para todo o X
    previsao_demissao_completa = smote_pipeline.predict(X)

    # Reconstruir o DataFrame original filtrado com as novas colunas de previsão
    df_resultados = df_hc_filtered_full.copy()  # DataFrame com todas as colunas originais filtradas
    
    # Garantir que o `target` original (y) seja adicionado ao df_resultados
    df_resultados['target_real'] = y.values  # Adiciona o target real para comparação

    df_resultados['prob_demissao'] = prob_demissao_completa
    df_resultados['previsao_demissao'] = previsao_demissao_completa

    logger.info("✓ Probabilidades e previsões calculadas para todos os colaboradores do conjunto filtrado.")

    # 5.2. Segmentando colaboradores em grupos de risco (Baixo, Médio, Alto)
    threshold_baixo = 0.3
    threshold_alto = 0.7

    def classificar_risco(prob):
        if prob >= threshold_alto:
            return 'Alto Risco'
        elif prob >= threshold_baixo:
            return 'Médio Risco'
        else:
            return 'Baixo Risco'

    df_resultados['segmento_risco'] = df_resultados['prob_demissao'].apply(classificar_risco)
    logger.info("✓ Colaboradores segmentados. Distribuição:\n" + df_resultados['segmento_risco'].value_counts().to_string())

    # 5.3. Exportando resultados de previsões e risco para 'df_previsoes_risco_ul.xlsx' com formatação
    # Selecionar apenas colunas que existem no DataFrame
    colunas_disponiveis = df_resultados.columns.tolist()
    
    # Definir limiares de risco (mesmos usados em 5.2)
    threshold_baixo_risco = 0.3
    threshold_alto_risco = 0.7
    
    # Definir colunas desejadas (na ordem)
    colunas_prioritarias = [
        'registro' ,'nome', 'cpf', 'cargo', 'cargo_completo', 'departamento', 'empresa_resumo', 'idade', 'salario_total', 
        'dias_trabalhado', 'dias_afastado', 'custo_afastamento', 'filhos', 'data_rescisao', 'treinamentos',
        'target_real', 'prob_demissao', 'previsao_demissao', 'segmento_risco'
    ]
    
    # Manter apenas colunas que existem
    cols_to_export = [col for col in colunas_prioritarias if col in colunas_disponiveis]
    
    # Adicionar outras colunas que existem mas não estão na lista prioritária
    outras_colunas = [col for col in colunas_disponiveis if col not in cols_to_export]
    cols_to_export.extend(outras_colunas)
    
    logger.info(f"✓ Colunas para exportação ({len(cols_to_export)}): {cols_to_export[:10]}...")

    df_final_export = df_resultados[cols_to_export].copy()

    # Criar workbook com formatação
    wb = Workbook()
    ws = wb.active
    ws.title = "Previsões de Risco de Demissão"

    # Escrever cabeçalhos com formatação
    header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
    header_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')
    header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

    for col_idx, header in enumerate(df_final_export.columns):
        cell = ws.cell(row=1, column=col_idx + 1, value=header)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_alignment

    # Escrever os dados
    for r_idx, row in enumerate(dataframe_to_rows(df_final_export, index=False, header=False), 2):
        for c_idx, value in enumerate(row):
            cell = ws.cell(row=r_idx, column=c_idx + 1, value=value)
            col_name = df_final_export.columns[c_idx]
            
            # --- INÍCIO DA ALTERAÇÃO ---
            # Formatação específica por coluna
            
            # 1. Formatação de DATA
            # Verifica se é objeto de data ou se o nome da coluna sugere data
            if isinstance(value, (datetime, pd.Timestamp)) or \
               any(x in col_name.lower() for x in ['data', 'nascimento', 'admissao', 'rescisao', 'ultimo_reajuste_individual', 'ultimo_reajuste_coletivo', 'exp_45dias', 'exp_90dias']):
                try:
                    cell.number_format = 'DD/MM/YYYY' # Formato Brasileiro
                except:
                    pass

            # 2. Demais formatações (Mantendo o que você já tinha)
            elif col_name == 'prob_demissao':
                cell.number_format = '0.00%'
            elif col_name in ['target_real', 'previsao_demissao']:
                cell.number_format = '0'
            elif col_name in ['salario_total', 'custo_afastamento']:
                cell.number_format = '#,##0.00'
            elif col_name in ['treinamentos']:
                cell.number_format = '0.00'
            elif col_name in ['idade', 'dias_trabalhado', 'dias_afastado']:
                cell.number_format = '0'
            # --- FIM DA ALTERAÇÃO ---
            
            # Alinhamento
            if col_name in ['target_real', 'prob_demissao', 'previsao_demissao', 'segmento_risco']:
                cell.alignment = Alignment(horizontal='center', vertical='center')

    # Ajustar a largura das colunas
    for col in ws.columns:
        max_length = 0
        column_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                if len(str(cell.value)) > max_length:
                    max_length = len(str(cell.value))
            except:
                pass
        adjusted_width = min((max_length + 2), 50)  # Máximo de 50 caracteres
        ws.column_dimensions[column_letter].width = adjusted_width

    # Congelar primeira linha
    ws.freeze_panes = 'A2'

    wb.save(OUTPUT_EXCEL_PREVISOES_FILE)
    logger.info(f"✓ Resultados exportados para '{OUTPUT_EXCEL_PREVISOES_FILE}' com formatação aprimorada.")
    logger.info(f"  Total de linhas: {len(df_final_export)}")
    logger.info(f"  Total de colunas: {len(df_final_export.columns)}")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 5 (Previsões e Exportação): {e}")
    import traceback
    traceback.print_exc()
    sys.exit("Script encerrado devido ao erro nas previsões ou exportação.")

gc.collect()  # Limpeza de memória
logger.info("✓ GC.collect() executado após Seção 5.")

# ==============================================================================
# 6. RECOMENDAÇÕES ESTRATÉGICAS
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 6 - RECOMENDAÇÕES ESTRATÉGICAS")
logger.info("="*80)

# Variáveis para armazenar texto das recomendações (sem fatores de risco/proteção)
recomendacoes_baixo_risco = ""
recomendacoes_medio_risco = ""
recomendacoes_alto_risco = ""
recomendacoes_kpis = ""

try:
    logger.info("\n--- 6.1. Ações Recomendadas por Segmento de Risco ---")
    recomendacoes_baixo_risco = """
    <h5 class="text-success">Colaboradores de Baixo Risco:</h5>
    <ul>
        <li>Manter engajamento através de reconhecimento, feedback positivo e oportunidades de desenvolvimento contínuo.</li>
        <li>Incentivar a mentoria e o compartilhamento de conhecimento.</li>
        <li>Coletar feedback regularmente (e.g., micro-surveys) para identificar potenciais problemas antes que escalem.</li>
        <li>Identificar "embaixadores" para cultura organizacional e mentores para novos colaboradores.</li>
    </ul>
    """
    recomendacoes_medio_risco = """
    <h5 class="text-warning">Colaboradores de Médio Risco:</h5>
    <ul>
        <li>Implementar 'stay interviews' para entender expectativas e motivações.</li>
        <li>Programas de retenção personalizados: planos de carreira, incentivos, flexibilidade.</li>
        <li>Treinamentos orientados ao desenvolvimento de habilidades críticas.</li>
        <li>Conexão com líderes e mentores para fortalecer pertencimento e identificar pontos de atrito.</li>
        <li>Revisão de carga de trabalho e equilíbrio vida-trabalho.</li>
    </ul>
    """
    recomendacoes_alto_risco = """
    <h5 class="text-danger">Colaboradores de Alto Risco:</h5>
    <ul>
        <li>Intervenção imediata e personalizada por RH e liderança direta.</li>
        <li>Investigar causas raízes (sobrecarga, conflitos, falta de perspectiva, saúde, insatisfação salarial).</li>
        <li>Planos de ação de curto prazo (realocação, projeto, ajuste de remuneração, suporte psicológico/social).</li>
        <li>Plano de sucessão caso retenção não seja viável, para mitigar impactos.</li>
        <li>Acompanhamento contínuo e feedback estruturado.</li>
    </ul>
    """

    logger.info("\n--- 6.2. KPIs de Monitoramento Contínuo ---")
    recomendacoes_kpis = """
    <h5 class="text-info">KPIs de Monitoramento Contínuo:</h5>
    <ul>
        <li>Taxa de Rotatividade Voluntária (geral e por segmento de risco).</li>
        <li>Custo de Demissão (custo de reposição, treinamento, perda de produtividade).</li>
        <li>Taxa de Engajamento (pesquisas de clima e pulso).</li>
        <li>% de retenção nos segmentos de Médio e Alto Risco após intervenções.</li>
        <li>Eficácia das ações de retenção (feedbacks qualitativos e quantitativos sobre as intervenções).</li>
        <li>Produtividade por segmento de risco.</li>
    </ul>
    """
    logger.info("\n✓ SEÇÃO 6 CONCLUÍDA COM SUCESSO")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 6 (Recomendações Estratégicas): {e}")
    sys.exit("Script encerrado devido ao erro nas recomendações estratégicas.")

# ==============================================================================
# 7. GERAÇÃO DO RELATÓRIO HTML COMPLETO
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 7 - GERAÇÃO DO RELATÓRIO HTML COMPLETO")
logger.info("="*80)

def preparar_arquivo_para_download(caminho_arquivo, pasta_html):
    # Garante que o arquivo esteja na mesma pasta do HTML e retorna apenas o nome do arquivo
    if not os.path.isfile(caminho_arquivo):
        logger.warning(f"Arquivo para download não encontrado: {caminho_arquivo}")
        return os.path.basename(caminho_arquivo)
    nome = os.path.basename(caminho_arquivo)
    destino = os.path.join(pasta_html, nome)
    # Evita copiar se o arquivo já está no destino e é o mesmo arquivo (caminho absoluto)
    if os.path.abspath(caminho_arquivo) != os.path.abspath(destino):
        os.makedirs(pasta_html, exist_ok=True)
        shutil.copy2(caminho_arquivo, destino)
        logger.info(f"Arquivo copiado para a pasta do HTML: {destino}")
    return nome  # sempre retornar só o nome do arquivo

html_dir = os.path.dirname(OUTPUT_HTML_FILE) if os.path.dirname(OUTPUT_HTML_FILE) else os.getcwd()
os.makedirs(html_dir, exist_ok=True)

link_previsoes = preparar_arquivo_para_download(OUTPUT_EXCEL_PREVISOES_FILE, html_dir)
link_glm_coef = preparar_arquivo_para_download(OUTPUT_GLM_COEF_CSV, html_dir)
link_glm_sum = preparar_arquivo_para_download(OUTPUT_GLM_SUMMARY_TXT, html_dir)
link_dashboard_excel = preparar_arquivo_para_download(OUTPUT_DASHBOARD_EXCEL_FILE, html_dir)

html_content = "" # Inicializar para garantir que existe

try:
    # 7.1. Preparar dados para o HTML
    # Garantir que todas as variáveis necessárias existam ou tenham um valor padrão
    # EDA
    eda_desc_stats_html = gerar_tabela_html(eda_desc_stats_df if 'eda_desc_stats_df' in locals() else pd.DataFrame(), "Estatísticas Descritivas (Variáveis Numéricas)")
    eda_target_dist_html = gerar_tabela_html(eda_target_dist_df if 'eda_target_dist_df' in locals() else pd.DataFrame(), "Distribuição da Variável Alvo")
    eda_corr_matrix_html = gerar_tabela_html(eda_corr_matrix_df if 'eda_corr_matrix_df' in locals() else pd.DataFrame(), "Matriz de Correlação Completa")
    eda_corr_target_html = gerar_tabela_html(eda_corr_target_df if 'eda_corr_target_df' in locals() else pd.DataFrame(), "Correlação com Target")

    eda_hist_box_imgs_html = "".join([f'<div class="col-md-6 mb-4"><img src="{converter_grafico_para_base64(os.path.join(OUTPUT_DIR, f"distribuicao_{col}.png"))}" class="img-fluid rounded shadow-sm" alt="Histograma/Boxplot de {col}"></div>' for col in numerical_cols[:3] if os.path.exists(os.path.join(OUTPUT_DIR, f"distribuicao_{col}.png"))])
    eda_cat_target_imgs_html = "".join([f'<div class="col-md-6 mb-4"><img src="{converter_grafico_para_base64(os.path.join(OUTPUT_DIR, f"distribuicao_categorica_{col}.png"))}" class="img-fluid rounded shadow-sm" alt="Distribuição Categórica {col} por Target"></div>' for col in categorical_cols[:3] if os.path.exists(os.path.join(OUTPUT_DIR, f"distribuicao_categorica_{col}.png"))])

    # Modelo Scikit-learn
    metrics_cards_skl = f"""
    {gerar_card_metricas("Acurácia", f"{accuracy_skl:.2%}", "Proporção de previsões corretas.")}
    {gerar_card_metricas("Precisão", f"{precision_skl:.2%}", "Das previsões 'INATIVO', quantas estavam corretas.")}
    {gerar_card_metricas("Recall", f"{recall_skl:.2%}", "Dos 'INATIVOS' reais, quantos foram identificados.")}
    {gerar_card_metricas("F1-Score", f"{f1_skl:.2f}", "Média harmônica entre precisão e recall.")}
    {gerar_card_metricas("ROC AUC", f"{roc_auc_skl:.2f}", "Probabilidade de o modelo classificar um positivo aleatório mais alto que um negativo aleatório.")}
    {gerar_card_metricas("Gini", f"{gini:.2f}", "Medida da desigualdade na capacidade de previsão do modelo (2*AUC-1).")}
    {gerar_card_metricas("Threshold Ajustado", f"{optimal_threshold:.2f}", "Ponto de corte para classificar como 'INATIVO'.")}
    {gerar_card_metricas("CV ROC AUC Média", f"{cv_scores_mean:.2f}", f"Média da AUC em validação cruzada (std: {cv_scores_std:.2f}).")}
    """

    # Modelo GLM
    # Badge e texto explicativo dinâmicos
    glm_badge_html = "<span class='badge bg-success ms-2'>Dados reais (p-values disponíveis)</span>"
    glm_table_title = f"Coeficientes GLM {glm_badge_html}"
    glm_table_caption = (
        "Modelo com dados exibindo coeficientes, erros padrão, "
        "p-values e Odds Ratios. As variáveis são ordenadas por significância (menor p-value)."
    )
    
    # Importante: exibir TODOS os resultados (sem limite)
    glm_summary_html = gerar_tabela_html(
        glm_summary_df if 'glm_summary_df' in locals() and not glm_summary_df.empty else pd.DataFrame(),
        title=glm_table_title,
        caption=glm_table_caption,
        limit=None
    )
    
    # Cards de métricas do GLM
    glm_metrics_cards = f"""
    {gerar_card_metricas("Log-Likelihood", f"{glm_loglike:.2f}" if isinstance(glm_loglike, (int, float)) else str(glm_loglike), "Medida de ajuste do modelo. Quanto maior (menos negativo), melhor.")}
    {gerar_card_metricas("AIC", f"{glm_aic:.2f}" if isinstance(glm_aic, (int, float)) else str(glm_aic), "Critério de Informação de Akaike. Penaliza complexidade. Menor é melhor.")}
    {gerar_card_metricas("BIC", f"{glm_bic:.2f}" if isinstance(glm_bic, (int, float)) else str(glm_bic), "Critério de Informação Bayesiano. Penaliza complexidade mais que AIC. Menor é melhor.")}
    """
    
    # Segmentação
    segmento_risco_dist_df = df_resultados['segmento_risco'].value_counts().to_frame(name='Contagem')
    segmento_risco_dist_html = gerar_tabela_html(segmento_risco_dist_df, "Distribuição por Segmento de Risco")

    # 7.2. Construção do HTML
    html_content = f"""
<!DOCTYPE html>
<html lang="pt-BR">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Relatório de Modelo Preditivo de Turnover com Regressão Logística e GLM</title>
    <!-- Bootstrap CSS -->
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <!-- Font Awesome para ícones -->
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0-beta3/css/all.min.css">
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f8f9fa; color: #343a40; }}
        .navbar {{ background-color: #0070c0 !important; }}
        .navbar-brand, .nav-link {{ color: #ffffff !important; }}
        .navbar-nav .nav-link.active {{ font-weight: bold; }}
        .header {{ background-color: #0070c0; color: white; padding: 4rem 0; text-align: center; }}
        .section-title {{ color: #0070c0; border-bottom: 3px solid #0070c0; padding-bottom: 10px; margin-bottom: 30px; }}
        .card {{ border-radius: 0.75rem; border: none; box-shadow: 0 0.5rem 1rem rgba(0,0,0,0.05); }}
        .card-title {{ color: #0070c0; font-weight: bold; }}
        .table-responsive {{ margin-top: 20px; }}
        .table {{ margin-bottom: 0; }}
        .table th {{ background-color: #e9ecef; color: #495057; }}
        .footer {{ background-color: #343a40; color: white; padding: 2rem 0; text-align: center; margin-top: 4rem; }}
        .img-fluid {{ max-width: 100%; height: auto; display: block; margin-left: auto; margin-right: auto; }}
        .back-to-top {{
            position: fixed;
            bottom: 20px;
            right: 20px;
            background-color: #0070c0;
            color: white;
            width: 40px;
            height: 40px;
            border-radius: 50%;
            text-align: center;
            line-height: 40px;
            font-size: 1.5rem;
            box-shadow: 0 2px 5px rgba(0,0,0,0.2);
            z-index: 1000;
            display: none; /* Escondido por padrão */
        }}
    </style>
</head>
<body data-bs-spy="scroll" data-bs-target="#navbar-example" data-bs-offset="50">

    <nav class="navbar navbar-expand-lg navbar-dark fixed-top shadow" id="navbar-example">
        <div class="container">
            <a class="navbar-brand" href="#">People Analytics</a>
            <button class="navbar-toggler" type="button" data-bs-toggle="collapse" data-bs-target="#navbarNav" aria-controls="navbarNav" aria-expanded="false" aria-label="Toggle navigation">
                <span class="navbar-toggler-icon"></span>
            </button>
            <div class="collapse navbar-collapse" id="navbarNav">
                <ul class="navbar-nav ms-auto">
                    <li class="nav-item"><a class="nav-link" href="#resumo">Resumo</a></li>
                    <li class="nav-item"><a class="nav-link" href="#contexto">Contexto</a></li>
                    <li class="nav-item"><a class="nav-link" href="#eda">EDA</a></li>
                    <li class="nav-item"><a class="nav-link" href="#modelo_skl">Modelo Scikit-learn</a></li>
                    <li class="nav-item"><a class="nav-link" href="#modelo_glm">Análise GLM</a></li>
                    <li class="nav-item"><a class="nav-link" href="#previsoes">Previsões</a></li>
                    <li class="nav-item"><a class="nav-link" href="#dashboard">Dashboard Executivo</a></li>
                    <li class="nav-item"><a class="nav-link" href="#recomendacoes">Recomendações</a></li>
                    <li class="nav-item"><a class="nav-link" href="#conclusao">Conclusão</a></li>
                </ul>
            </div>
        </div>
    </nav>

    <div class="header">
        <div class="container">
            <h1 class="display-4">Relatório de People Analytics</h1>
            <p class="lead">Modelo Preditivo de Turnover com Regressão Logística e GLM</p>
            <p class="mt-3"><i>Gerado em: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}</i></p>
        </div>
    </div>

    <div class="container my-5">
        <section id="resumo" class="mb-5">
            <h2 class="section-title">1. Resumo Executivo</h2>
            <p>Desenvolvimento de um modelo de Machine Learning para prever a probabilidade de desligamento voluntário (churn) de colaboradores, permitindo ações preventivas de retenção baseadas em dados. A solução integra um modelo de Regressão Logística (Scikit-learn) para alta acurácia preditiva e um Modelo Linear Generalizado (Statsmodels GLM) para interpretabilidade estatística. O objetivo é fornecer insights acionáveis para a Gestão de Pessoas.</p>
            <p>O modelo Scikit-learn alcançou uma <strong>ROC AUC de {roc_auc_skl:.2f}</strong>, indicando uma excelente capacidade preditiva, e um <strong>Coeficiente de Gini de {gini:.2f}</strong>. O GLM, por sua vez, revelou os principais fatores de risco e proteção com base em sua significância estatística (p-values). Os resultados incluem a segmentação de colaboradores por risco (Baixo, Médio, Alto) e um dashboard executivo para facilitar a tomada de decisão.</p>
            <div class="row">{metrics_cards_skl}</div>
        </section>

        <section id="contexto" class="mb-5">
            <h2 class="section-title">2. Contexto e Objetivos</h2>
            <p>A rotatividade de colaboradores (turnover) é um desafio significativo para qualquer organização, impactando custos de recrutamento, treinamento e produtividade. Este projeto visa utilizar técnicas avançadas de Machine Learning para:</p>
            <ul>
                <li><strong>Prever</strong> a probabilidade de demissão de um colaborador com alta precisão.</li>
                <li><strong>Identificar</strong> os principais fatores (variáveis) que influenciam a demissão e sua significância estatística.</li>
                <li><strong>Segmentar</strong> colaboradores em diferentes níveis de risco (Baixo, Médio, Alto) para intervenções personalizadas.</li>
                <li><strong>Fornecer</strong> recomendações estratégicas e um dashboard executivo para ações proativas de retenção.</li>
            </ul>
            <p>Os dados utilizados incluem informações de atestados médicos e dados cadastrais de colaboradores da unidade 'UL', filtrados para o período de 2022.</p>
        </section>

        <section id="eda" class="mb-5">
            <h2 class="section-title">3. Análise Exploratória dos Dados (EDA)</h2>
            <p>A EDA foi realizada para entender a estrutura, distribuição e relações entre as variáveis. Esta etapa é fundamental para identificar padrões, tratar anomalias e preparar os dados para a modelagem.</p>

            <h3 class="mt-4">3.1. Estatísticas Descritivas</h3>
            <p>Visão geral das principais estatísticas das variáveis numéricas:</p>
            {eda_desc_stats_html}

            <h3 class="mt-4">3.2. Distribuição da Variável Alvo ('target')</h3>
            <p>A variável alvo (target) indica se o colaborador está ATIVO (0) ou INATIVO (1 - demitido). A distribuição mostra um desbalanceamento, com uma minoria de casos 'INATIVO'.</p>
            {eda_target_dist_html}
            <div class="row">
                <div class="col-md-8 mx-auto mb-4">
                    <img src="{eda_target_dist_base64}" class="img-fluid rounded shadow-sm" alt="Distribuição da Variável Alvo">
                    <p class="text-center text-muted mt-2"><i>Distribuição da variável alvo (0=Ativo, 1=Inativo).</i></p>
                </div>
            </div>

            <h3 class="mt-4">3.3. Análise de Correlação</h3>
            <p>O heatmap abaixo ilustra as correlações entre as variáveis numéricas, incluindo a variável 'target'. Coeficientes próximos de 1 ou -1 indicam forte correlação positiva ou negativa, respectivamente. Coeficientes próximos de 0 indicam pouca ou nenhuma correlação linear.</p>
            {eda_corr_matrix_html}
            {eda_corr_target_html}
            <div class="row">
                <div class="col-md-8 mx-auto mb-4">
                    <img src="{eda_corr_heatmap_base64}" class="img-fluid rounded shadow-sm" alt="Heatmap de Correlação">
                    <p class="text-center text-muted mt-2"><i>Heatmap da Matriz de Correlação entre variáveis numéricas.</i></p>
                </div>
            </div>

            <h3 class="mt-4">3.4. Visualizações Detalhadas</h3>
            <p>Exemplos de distribuições para algumas variáveis:</p>
            <div class="row">{eda_hist_box_imgs_html}</div>
            <div class="row">{eda_cat_target_imgs_html}</div>

        </section>

        <section id="modelo_skl" class="mb-5">
            <h2 class="section-title">4. Modelo de Machine Learning: Regressão Logística (Scikit-learn)</h2>
            <p>Um Pipeline de Regressão Logística foi construído utilizando Scikit-learn, incluindo pré-processamento (padronização e One-Hot Encoding). Este modelo é otimizado para a performance preditiva.</p>

            <h3 class="mt-4">4.1. Configuração do Pipeline</h3>
            <p>O pipeline é composto por:</p>
            <ul>
                <li><code>ColumnTransformer</code>: Aplica <code>StandardScaler</code> às variáveis numéricas e <code>OneHotEncoder</code> às categóricas para convertê-las em formato numérico.</li>
                <li><code>LogisticRegression</code>: O modelo de classificação final. Utilizamos o hiperparâmetro <code\class_weight='balanced'</code> para lidar com o desbalanceamento das classes, ajustando o modelo para dar maior importância à detecção de demissões sem alterar a distribuição real dos dados.</li>
            </ul>
            
            <h3 class="mt-4">4.2. Matriz de Confusão</h3>
            <p>A matriz de confusão detalha o desempenho do modelo, mostrando o número de verdadeiros positivos, verdadeiros negativos, falsos positivos e falsos negativos. Valores ideais são altos nas diagonais principais.</p>
            <div class="row">
                <div class="col-md-8 mx-auto mb-4">
                    <img src="{confusion_matrix_base64}" class="img-fluid rounded shadow-sm" alt="Matriz de Confusão">
                    <p class="text-center text-muted mt-2"><i>Matriz de Confusão do Modelo de Regressão Logística (Scikit-learn) com threshold ajustado.</i></p>
                </div>
            </div>

            <h3 class="mt-4">4.3. Curva ROC e Coeficiente de Gini</h3>
            <p>A Curva ROC (Receiver Operating Characteristic) é uma ferramenta gráfica para avaliar o desempenho de modelos de classificação. A Área Sob a Curva (AUC) mede a capacidade do modelo de distinguir entre as classes (quanto mais próximo de 1, melhor). O Coeficiente de Gini é derivado da AUC (<code>2 * AUC - 1</code>) e varia de 0 a 1, onde 1 é um modelo perfeito.</p>
            <div class="row">
                <div class="col-md-8 mx-auto mb-4">
                    <img src="{roc_curve_base64}" class="img-fluid rounded shadow-sm" alt="Curva ROC">
                    <p class="text-center text-muted mt-2"><i>Curva ROC com valor AUC e Coeficiente de Gini.</i></p>
                </div>
            </div>
            
            <h3 class="mt-4">4.4. Relatório de Classificação</h3>
            <pre class="bg-light p-3 rounded">{classification_report_str}</pre>
            
        </section>

        <section id="modelo_glm" class="mb-5">
            <h2 class="section-title">5. Análise do Modelo GLM (Statsmodels)</h2>
            <p>O Modelo Linear Generalizado (GLM) foi utilizado para fornecer uma interpretação estatística mais aprofundada dos coeficientes e sua significância (p-values). Este modelo foi treinado com técnicas de limpeza para garantir estabilidade e interpretabilidade, ajudando a identificar quais variáveis são estatisticamente importantes na previsão de demissão.</p>
            <div class="row">{glm_metrics_cards}</div>
            
            <h3 class="mt-4">5.1. Coeficientes do Modelo GLM — Fatores de Risco e Proteção</h3>
            <p>O GLM fornece os coeficientes (Beta) que representam o impacto de cada variável na probabilidade de demissão, e as Odds Ratios, que quantificam essa relação. Os p-values indicam a significância estatística de cada fator.</p>
            {glm_summary_html}
            <p class="text-muted mt-2">
                <i>**Significância:** *** p &lt; 0.001, ** p &lt; 0.01, * p &lt; 0.05.</i><br>
                <i>Odds Ratio &gt; 1: Aumenta a chance de demissão. Odds Ratio &lt; 1: Diminui a chance de demissão.</i>
            </p>
            <a href="{link_glm_coef}" class="btn btn-primary mt-3" download><i class="fas fa-download"></i> Baixar Coeficientes GLM (CSV)</a>
            <a href="{link_glm_sum}" class="btn btn-secondary mt-3 ms-2" download><i class="fas fa-download"></i> Baixar Sumário GLM (TXT)</a>
        </section>

        <section id="previsoes" class="mb-5">
            <h2 class="section-title">6. Previsões e Segmentação de Colaboradores</h2>
            <p>O modelo Scikit-learn foi aplicado a todos os colaboradores do conjunto filtrado para calcular a probabilidade de demissão e segmentá-los em grupos de risco (Baixo, Médio, Alto). Essa segmentação é a base para as ações proativas do RH.</p>

            <h3 class="mt-4">6.1. Distribuição por Segmento de Risco</h3>
            <p>Esta tabela mostra quantos colaboradores foram classificados em cada categoria de risco com base nas suas probabilidades de demissão.</p>
            {segmento_risco_dist_html}
            <p class="text-muted mt-2"><i>Limiares utilizados: Prob. &lt; {threshold_baixo_risco:.2f} = Baixo Risco; {threshold_baixo_risco:.2f} &le; Prob. &lt; {threshold_alto_risco:.2f} = Médio Risco; Prob. &ge; {threshold_alto_risco:.2f} = Alto Risco.</i></p>

            <h3 class="mt-4">6.2. Exportação Detalhada</h3>
            <p>Um arquivo Excel detalhado (<code>df_previsoes_risco_ul.xlsx</code>) foi gerado contendo todas as informações originais dos colaboradores, suas probabilidades de demissão, a previsão do modelo e o segmento de risco ao qual pertencem. Este arquivo serve como um registro completo para análise individual.</p>
            <a href="{link_previsoes}" class="btn btn-success mt-3" download><i class="fas fa-file-excel"></i> Baixar Relatório de Previsões (Excel)</a>
        </section>
        
        <section id="dashboard" class="mb-5">
            <h2 class="section-title">7. Dashboard Executivo de Gestão de Pessoas</h2>
            <p>Para facilitar a tomada de decisão, um Dashboard em Excel foi criado, consolidando as principais informações. Ele contém abas com:</p>
            <ul>
                <li><strong>Resumo Executivo:</strong> Métricas chave e distribuição de risco.</li>
                <li><strong>Alto Risco:</strong> Lista detalhada de colaboradores que demandam ação imediata.</li>
                <li><strong>Médio Risco:</strong> Lista de colaboradores para programas de retenção.</li>
                <li><strong>Fatores de Risco:</strong> Variáveis que aumentam a chance de demissão (do GLM).</li>
                <li><strong>Fatores de Proteção:</strong> Variáveis que diminuem a chance de demissão (do GLM).</li>
            </ul>
            <p>Este dashboard é a principal ferramenta para o RH e a liderança acessarem os insights e agirem proativamente.</p>
            <a href="{link_dashboard_excel}" class="btn btn-info mt-3" download><i class="fas fa-file-excel"></i> Baixar Dashboard Completo (Excel)</a>
        </section>

        <section id="recomendacoes" class="mb-5">
            <h2 class="section-title">8. Recomendações Estratégicas</h2>
            <p>Com base na segmentação de risco e nos fatores identificados pelos modelos, seguem recomendações práticas para a Gestão de Pessoas.</p>

            <h3 class="mt-4">8.1. Ações por Segmento de Risco</h3>
            {recomendacoes_baixo_risco}
            {recomendacoes_medio_risco}
            {recomendacoes_alto_risco}
        
            <h3 class="mt-4">8.2. KPIs de Monitoramento Contínuo</h3>
            <p>Para garantir a eficácia das intervenções e o sucesso do programa de retenção, é crucial monitorar os seguintes indicadores:</p>
            {recomendacoes_kpis}
        </section>

        <section id="conclusao" class="mb-5">
            <h2 class="section-title">9. Conclusão e Próximos Passos</h2>
            <p>Este projeto demonstra o poder de Machine Learning na identificação proativa de riscos de demissão. O modelo híbrido desenvolvido oferece insights valiosos sobre os fatores que influenciam a decisão de um colaborador permanecer na empresa e permite a segmentação para ações de retenção personalizadas, tornando-o uma ferramenta estratégica essencial para a Gestão de Pessoas.</p>
            <p><strong>Próximos Passos Sugeridos:</strong></p>
            <ul>
                <li><strong>Validação de Negócio:</strong> Discutir os fatores de risco e proteção identificados com especialistas de RH e lideranças para validar as descobertas no contexto da empresa.</li>
                <li><strong>Implementação de Ações:</strong> Desenvolver e implementar planos de ação detalhados e personalizados para os segmentos de médio e alto risco.</li>
                <li><strong>Integração em Processos:</strong> Integrar o modelo em um processo de monitoramento regular, talvez com atualizações mensais ou trimestrais dos dados e do modelo.</li>
                <li><strong>Refinamento Contínuo:</strong> Explorar variáveis adicionais (e.g., dados de performance, feedback de clima, interações com gestores) ou outros modelos de Machine Learning para aprimorar ainda mais a precisão preditiva e a riqueza dos insights.</li>
                <li><strong>Monitoramento de Impacto:</strong> Medir o impacto das ações de retenção nos KPIs definidos, criando um ciclo de feedback para otimização contínua.</li>
            </ul>
        </section>

    </div>

    <a href="#" class="back-to-top btn rounded-circle"><i class="fas fa-arrow-up"></i></a>

    <footer class="footer">
        <div class="container">
            <p>&copy; {datetime.now().year} Rodrigo Bernardes. Todos os direitos reservados.</p>
        </div>
    </footer>

    <!-- Bootstrap JS (com Popper.js) -->
    <script src="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/js/bootstrap.bundle.min.js"></script>
    <script>
        // Script para rolagem suave e botão voltar ao topo
        document.addEventListener('DOMContentLoaded', function() {{
            // Smooth scrolling for internal links
            document.querySelectorAll('a[href^="#"]').forEach(anchor => {{
                anchor.addEventListener('click', function (e) {{
                    e.preventDefault();
                    document.querySelector(this.getAttribute('href')).scrollIntoView({{
                        behavior: 'smooth'
                    }});
                }});
            }});

            // Back to top button functionality
            const backToTopButton = document.querySelector('.back-to-top');
            window.addEventListener('scroll', () => {{
                if (window.scrollY > 300) {{ // Show button after scrolling 300px
                    backToTopButton.style.display = 'block';
                }} else {{
                    backToTopButton.style.display = 'none';
                }}
            }});

            backToTopButton.addEventListener('click', () => {{
                window.scrollTo({{ top: 0, behavior: 'smooth' }});
            }});
        }});
    </script>
</body>
</html>
"""
    # 7.3. Salvar o HTML em um arquivo
    with open(OUTPUT_HTML_FILE, 'w', encoding='utf-8') as f:
        f.write(html_content)
    logger.info(f"✓ Relatório HTML gerado com sucesso em '{OUTPUT_HTML_FILE}'")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 7 (Geração do Relatório HTML): {e}")
    import traceback
    traceback.print_exc()

# ==============================================================================
# 9. DASHBOARD DE GESTÃO DE PESSOAS
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 9 - DASHBOARD DE GESTÃO DE PESSOAS (FORMATADO)")
logger.info("="*80)

try:
    if df_resultados.empty:
        raise ValueError("DataFrame de resultados vazio. Não é possível gerar o Dashboard.")
    
    # --- Função Auxiliar de Formatação (Reutilizável) ---
    def formatar_aba(ws, df_source):
        # 1. Estilo do Cabeçalho
        header_font = Font(name='Calibri', size=11, bold=True, color='FFFFFF')
        header_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')
        header_alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

        # Aplica estilo na primeira linha
        for col_idx, header_cell in enumerate(ws[1], 1):
            header_cell.font = header_font
            header_cell.fill = header_fill
            header_cell.alignment = header_alignment

        # 2. Formatação de Dados (Datas, Percentuais, Moeda)
        for r_idx, row in enumerate(ws.iter_rows(min_row=2), 2):
            for c_idx, cell in enumerate(row):
                col_name = df_source.columns[c_idx] if c_idx < len(df_source.columns) else ""
                
                # DATA (dd/mm/aaaa)
                if isinstance(cell.value, (datetime, pd.Timestamp)) or \
                   any(x in str(col_name).lower() for x in ['data', 'nascimento', 'admissao', 'rescisao']):
                    cell.number_format = 'DD/MM/YYYY'
                
                # PERCENTUAL
                elif 'prob' in str(col_name).lower() or 'taxa' in str(col_name).lower():
                    # Se for string com %, tenta limpar, senão aplica formato numérico
                    if isinstance(cell.value, (int, float)):
                        cell.number_format = '0.00%'
                
                # MOEDA
                elif 'salario' in str(col_name).lower() or 'custo' in str(col_name).lower():
                    cell.number_format = '#,##0.00'

        # 3. Ajuste de Largura
        for col in ws.columns:
            max_length = 0
            column_letter = get_column_letter(col[0].column)
            for cell in col:
                try:
                    if cell.value: max_length = max(max_length, len(str(cell.value)))
                except: pass
            ws.column_dimensions[column_letter].width = min(max_length + 2, 60)
    # ----------------------------------------------------

    # 9.1. Análise por Segmento de Risco
    segmento_stats = df_resultados.groupby('segmento_risco').agg({
        'prob_demissao': ['count', 'mean', 'min', 'max'],
        'dias_afastado': 'mean',
        'salario_total': 'mean',
        'idade': 'mean',
        'dias_trabalhado': 'mean',
        'treinamentos': 'mean'
    }).round(2)
    segmento_stats.columns = ['_'.join(col).strip() for col in segmento_stats.columns.values]
    segmento_stats = segmento_stats.reset_index().rename(columns={'segmento_risco': 'Segmento de Risco'})

    # 9.2. Colaboradores de Alto Risco
    cols_to_show = ['registro', 'nome', 'cargo', 'cargo_completo', 'departamento', 'empresa_resumo', 
                    'data_rescisao', 'prob_demissao', 'idade', 'dias_trabalhado', 'salario_total', 'treinamentos',
                    'dias_afastado', 'target_real']
    
    alto_risco_df = df_resultados[df_resultados['segmento_risco'] == 'Alto Risco'].sort_values('prob_demissao', ascending=False).copy()
    alto_risco_df = alto_risco_df[[c for c in cols_to_show if c in alto_risco_df.columns]]

    # 9.3. Colaboradores de Médio Risco
    medio_risco_df = df_resultados[df_resultados['segmento_risco'] == 'Médio Risco'].sort_values('prob_demissao', ascending=False).copy()
    medio_risco_df = medio_risco_df[[c for c in cols_to_show if c in medio_risco_df.columns]]

    # 9.4 e 9.5. Fatores GLM
    if 'glm_summary_df' in locals() and not glm_summary_df.empty:
        fatores_risco_df = glm_summary_df[glm_summary_df['Odds Ratio'] > 1].sort_values('Odds Ratio', ascending=False).copy()
        fatores_protecao_df = glm_summary_df[glm_summary_df['Odds Ratio'] < 1].sort_values('Odds Ratio', ascending=True).copy()
    else:
        fatores_risco_df = pd.DataFrame(columns=['Feature', 'Odds Ratio', 'p-value'])
        fatores_protecao_df = pd.DataFrame(columns=['Feature', 'Odds Ratio', 'p-value'])

    # 9.6. GERAÇÃO DO ARQUIVO EXCEL
    wb_dashboard = Workbook()
    
    # ABA 1: Resumo
    ws_resumo = wb_dashboard.active
    ws_resumo.title = "Resumo Executivo"
    ws_resumo.append(['Relatório de People Analytics: Previsão de Demissão'])
    ws_resumo.append(['Gerado em:', datetime.now().strftime('%d/%m/%Y %H:%M:%S')])
    ws_resumo.append([''])
    ws_resumo.append(['Métrica', 'Valor'])
    ws_resumo.append(['Total Colaboradores', len(df_resultados)])
    ws_resumo.append(['Alto Risco', len(alto_risco_df)])
    ws_resumo.append(['Acurácia Modelo', f"{accuracy_skl:.2%}"])
    
    # ABA 2: Alto Risco
    ws_alto = wb_dashboard.create_sheet("Alto Risco")
    for r in dataframe_to_rows(alto_risco_df, index=False, header=True): ws_alto.append(r)
    formatar_aba(ws_alto, alto_risco_df) # <--- APLICA A FORMATAÇÃO AQUI

    # ABA 3: Médio Risco
    ws_medio = wb_dashboard.create_sheet("Médio Risco")
    for r in dataframe_to_rows(medio_risco_df, index=False, header=True): ws_medio.append(r)
    formatar_aba(ws_medio, medio_risco_df) # <--- APLICA A FORMATAÇÃO AQUI

    # ABA 4: Fatores de Risco
    ws_frisco = wb_dashboard.create_sheet("Fatores de Risco")
    for r in dataframe_to_rows(fatores_risco_df, index=False, header=True): ws_frisco.append(r)
    formatar_aba(ws_frisco, fatores_risco_df)

    # ABA 5: Fatores de Proteção
    ws_fprotecao = wb_dashboard.create_sheet("Fatores de Proteção")
    for r in dataframe_to_rows(fatores_protecao_df, index=False, header=True): ws_fprotecao.append(r)
    formatar_aba(ws_fprotecao, fatores_protecao_df)

    wb_dashboard.save(OUTPUT_DASHBOARD_EXCEL_FILE)
    logger.info(f"✓ Dashboard exportado com sucesso para '{OUTPUT_DASHBOARD_EXCEL_FILE}'")

except Exception as e:
    logger.error(f"ERRO na SEÇÃO 9: {e}")
    import traceback
    traceback.print_exc()

gc.collect()

# ==============================================================================
# 10. FINALIZAÇÃO E EXPORTAÇÕES (Sumário)
# ==============================================================================
logger.info("\n" + "="*80)
logger.info("SEÇÃO 10 - FINALIZAÇÃO E EXPORTAÇÕES")
logger.info("="*80)
logger.info("✓ Todas as etapas do projeto foram concluídas.")
logger.info(f"O relatório HTML completo está disponível em: {OUTPUT_HTML_FILE}")
logger.info(f"Os dados detalhados com previsões estão em: {OUTPUT_EXCEL_PREVISOES_FILE}")
logger.info(f"Os coeficientes do modelo GLM estão em: {OUTPUT_GLM_COEF_CSV}")
logger.info(f"O sumário completo do modelo GLM está em: {OUTPUT_GLM_SUMMARY_TXT}")
logger.info(f"O Dashboard Executivo em Excel está em: {OUTPUT_DASHBOARD_EXCEL_FILE}")
logger.info(f"Os gráficos PNG individuais estão no diretório: {OUTPUT_DIR}")

logger.info("\nPROJETO DE PEOPLE ANALYTICS CONCLUÍDO COM SUCESSO!")
logger.info("================================================================================")

# Controle de atualização de processo
tempo_1 = [id, datetime.today(), 1]
ws_p.append(tempo_1)
wb_p.save(path_registros_processos)

print('--------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('--------------------------------------------------------------------------------------------')

<>:1774: SyntaxWarning: invalid escape sequence '\c'
<>:1774: SyntaxWarning: invalid escape sequence '\c'
C:\Users\rodrigo.bernandes\AppData\Local\Temp\ipykernel_9604\4209434404.py:1774: SyntaxWarning: invalid escape sequence '\c'
  # 9. DASHBOARD DE GESTÃO DE PESSOAS
[26/06/2026 10:14:11] [INFO] ✓ Logger configurado com sucesso!
[26/06/2026 10:14:11] [INFO] ✓ Bibliotecas importadas e logger configurado com sucesso.
[26/06/2026 10:14:12] [INFO] ✓ Funções auxiliares HTML configuradas.
[26/06/2026 10:14:12] [INFO] 
[26/06/2026 10:14:12] [INFO] SEÇÃO 1 - CARREGAMENTO E PRÉ-PROCESSAMENTO INICIAL DOS DADOS
[26/06/2026 10:14:12] [INFO] ================================================================================
[26/06/2026 10:14:22] [INFO] ✓ Base de dados carregada com 9811 registros e 80 colunas.
[26/06/2026 10:14:22] [INFO] ✓ Coluna 'ano_rescisao' tratada para string.
[26/06/2026 10:14:22] [INFO] ✓ Coluna 'ano_atestado' tratada para string.
[26/06/2026 10:14:22] [INFO] ✓ Dados filtrado

--------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   0:00:39.317821

--------------------------------------------------------------------------------------------
